# EXPERIMENTAL — FC → SC direction

**This is an experimental copy** under `notebooks-FC_to_SC-experimental/`, translated to run the **FC → SC** direction (predict the structural connectome *from* functional).

Direction is controlled by the **`SOURCE, TARGET` toggle** in the first code cell below:
- `SOURCE, TARGET = "FC", "SC"`  → **FC → SC** (this notebook)
- `SOURCE, TARGET = "SC", "FC"`  → original SC → FC

The original SC→FC version lives in `notebooks/model_overviews/`. Everything else (models, eval) is the **shared** code — only the direction toggle differs.

# Cross-modal PCA/PLS closed-form overview

This notebook explains and tests the closed-form PCA/PLS models in `models/architectures/crossmodal_pca_pls.py`.

Covered models:

1. `CrossModalPCA`
2. `CrossModal_PLS_SVD`
3. `CrossModal_PCA_PLS`

The notebook is intentionally CPU-oriented for model execution. It uses the YAML defaults as the source of model parameters, references the production sbatch launchers, and builds the HCP data/loaders once for the shared `SC -> FC` local tests.

In [11]:
import importlib
from copy import deepcopy
from pathlib import Path

import pandas as pd
import torch
import yaml
from IPython.display import Markdown, display

import main
import models.registry
import models.eval.evaluator
import models.architectures.crossmodal_pca_pls

importlib.reload(models.architectures.crossmodal_pca_pls)
importlib.reload(models.eval.evaluator)
importlib.reload(models.registry)
importlib.reload(main)

from main import Sim
from models.architectures.utils import get_model_input
from models.registry import build_model, get_default_config, resolve_source_dependent_config

REPO_ROOT = Path.cwd()
RESULTS_ROOT = Path("results/local_results/crossmodal_pca_pls_closed_form_overview")
RESULTS_ROOT.mkdir(parents=True, exist_ok=True)

# ====================== DIRECTION TOGGLE ======================
# EXPERIMENTAL FC -> SC NOTEBOOK. Flip these two to switch direction:
#   FC -> SC  :  SOURCE, TARGET = "FC", "SC"   (this experimental notebook)
#   SC -> FC  :  SOURCE, TARGET = "SC", "FC"   (original direction)
SOURCE = "FC"
TARGET = "SC"
# ==============================================================
PARCELLATION = "Glasser"
SHUFFLE_SEED = 0
DATA_LOAD_MODE = "precomputed"

CLOSED_FORM_MODELS = {
    "CrossModalPCA": {
        "config": REPO_ROOT / "models/configs/CrossModalPCA.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModalPCA/tune_array_pca_SC_SCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModalPCA/tune_model_single_pca.sh",
        ],
    },
    "CrossModal_PLS_SVD": {
        "config": REPO_ROOT / "models/configs/CrossModal_PLS_SVD.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModal_PLS_SVD/tune_array_pls_svd_SC_SCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModal_PLS_SVD/tune_model_sequential_pls_svd_SC.sh",
        ],
    },
    "CrossModal_PCA_PLS": {
        "config": REPO_ROOT / "models/configs/CrossModal_PCA_PLS.yml",
        "sbatch": [
            REPO_ROOT / "sbatch/CrossModal_PCA_PLS/tune_array_pca_pls_SC_SCr2t_SCpSCr2t_seeds.sh",
            REPO_ROOT / "sbatch/CrossModal_PCA_PLS/tune_model_single_pca_pls.sh",
        ],
    },
}

def read_yaml(path):
    with open(path) as f:
        return yaml.safe_load(f)


def show_yaml_default(model_name):
    cfg = read_yaml(CLOSED_FORM_MODELS[model_name]["config"])
    display(Markdown(f"**`{model_name}` config**: `{CLOSED_FORM_MODELS[model_name]['config'].relative_to(REPO_ROOT)}`"))
    display(pd.Series(cfg.get("default", {}), dtype="object"))
    display(Markdown("Search-space keys: " + ", ".join(f"`{k}`" for k in cfg.get("search_space", {}))))


def show_sbatch_refs(model_name):
    refs = CLOSED_FORM_MODELS[model_name]["sbatch"]
    if not refs:
        display(Markdown("No dedicated sbatch launcher is currently present for this model; use `main.py` with its YAML config."))
        return
    lines = [f"- `{p.relative_to(REPO_ROOT)}`" for p in refs]
    display(Markdown("Production launcher references:\n" + "\n".join(lines)))


def model_config_for_notebook(model_name):
    cfg = get_default_config(model_name, path=str(CLOSED_FORM_MODELS[model_name]["config"]))
    cfg.setdefault("data", {})
    cfg["data"].update({
        "source": SOURCE,
        "target": TARGET,
        "parcellation": PARCELLATION,
        "shuffle_seed": SHUFFLE_SEED,
        "data_load_mode": DATA_LOAD_MODE,
    })
    cfg.setdefault("model", {})
    # Closed-form local inspection should stay on CPU. This is an execution-policy override,
    # not a model hyperparameter sweep value.
    cfg["model"]["device"] = "cpu"
    return resolve_source_dependent_config(cfg)


def build_closed_form_model(model_name, base):
    cfg = model_config_for_notebook(model_name)
    model_cfg = deepcopy(cfg["model"])
    model_cfg.pop("name", None)
    return build_model(base, model_name, model_cfg).eval()


def evaluate_closed_form_model(model_name, model):
    out = data_sim._evaluate_model(
        model,
        mode="dev",
        model_name=model_name,
        base=data_sim.base,
        train_loader=data_sim.train_loader,
        val_loader=data_sim.val_loader,
        test_loader=data_sim.test_loader,
    )
    return out


def inspect_one_batch(model, title):
    batch = next(iter(data_sim.val_loader))
    x = get_model_input(batch)
    y = batch["y"]
    with torch.no_grad():
        y_hat = model(x)
    print(title)
    print("device:", next(model.parameters(), torch.empty(0)).device if list(model.parameters()) else "buffer-only")
    print("x shape:", tuple(x.shape) if torch.is_tensor(x) else {k: tuple(v.shape) for k, v in x.items()})
    print("y shape:", tuple(y.shape))
    print("y_hat shape:", tuple(y_hat.shape))
    print("one-batch mse:", float(torch.mean((y_hat.cpu() - y.cpu()) ** 2)))
    return y_hat, y

## Production references and shared local data

The full production closed-form sweeps run through `main.py --use_tune` from the sbatch scripts under `sbatch/CrossModal_*`. Those launchers request GPUs as Ray resources in some cases because the shared experiment harness is GPU-capable, but the local inspection path here keeps the closed-form models on CPU.

To avoid repeated expensive loader construction, this notebook builds one `Sim` object for the shared `SC -> FC` data context and reuses its `base`, `train_loader`, `val_loader`, and `test_loader` for all three models.

In [ ]:
data_sim = Sim(
    model_name="CrossModal_PCA_PLS",
    config_path=str(CLOSED_FORM_MODELS["CrossModal_PCA_PLS"]["config"]),
    source=SOURCE,
    target=TARGET,
    parcellation=PARCELLATION,
    shuffle_seed=SHUFFLE_SEED,
    data_load_mode=DATA_LOAD_MODE,
)

batch = next(iter(data_sim.train_loader))
print("train / val / test batches:", len(data_sim.train_loader), len(data_sim.val_loader), len(data_sim.test_loader))
print("batch keys:", sorted(batch.keys()))
print("x shape:", tuple(get_model_input(batch).shape))
print("y shape:", tuple(batch["y"].shape))


## 1. `CrossModalPCA`

`CrossModalPCA` is the most direct PCA transfer baseline. It centers the source edge vector, projects into the first $k$ source PCA components, then decodes the same coordinates through the first $k$ target PCA components:

$$
z_s = (x_s - \mu_s)P_{s,k}
$$

$$
\hat y_t = \mu_t + z_s P_{t,k}^{T}.
$$

There is no learned cross-modal map. The model asks whether source PCA coordinates can be reused as target PCA coordinates.

In [ ]:
show_yaml_default("CrossModalPCA")
show_sbatch_refs("CrossModalPCA")

pca_model = build_closed_form_model("CrossModalPCA", data_sim.base)
pca_run = evaluate_closed_form_model("CrossModalPCA", pca_model)
inspect_one_batch(pca_model, "CrossModalPCA one-batch check")
pd.Series(pca_run["test_metrics"]["base_metrics"]).sort_index()


## 2. `CrossModal_PLS_SVD`

`CrossModal_PLS_SVD` works in raw edge space and estimates the dominant source-target cross-covariance directions without explicitly forming the full edge-by-edge covariance matrix.

For centered train matrices $X$ and $Y$, the implicit operator is

$$
C = X^T Y,
$$

with matrix-vector products

$$
Cv = X^T(Yv), \qquad C^T u = Y^T(Xu).
$$

The top singular vectors define source and target PLS directions:

$$
C \approx W_x S W_y^T.
$$

A latent least-squares map is then fit:

$$
Z_x = XW_x, \qquad Z_y = YW_y, \qquad B = \arg\min_B \|Z_xB - Z_y\|_F^2.
$$

Prediction is

$$
\hat y_t = \mu_t + (x_s - \mu_s)W_xBW_y^T.
$$

In [ ]:
show_yaml_default("CrossModal_PLS_SVD")
show_sbatch_refs("CrossModal_PLS_SVD")

pls_svd_model = build_closed_form_model("CrossModal_PLS_SVD", data_sim.base)
pls_svd_run = evaluate_closed_form_model("CrossModal_PLS_SVD", pls_svd_model)
inspect_one_batch(pls_svd_model, "CrossModal_PLS_SVD one-batch check")
print("singular values shape:", tuple(pls_svd_model.singular_values.shape))
pd.Series(pls_svd_run["test_metrics"]["base_metrics"]).sort_index()


## 3. `CrossModal_PCA_PLS`

`CrossModal_PCA_PLS` is the main closed-form PCA+PLS bridge. It first projects each source modality into PCA score space and concatenates source latents when the source is multi-input:

$$
z_{s_i} = (x_{s_i} - \mu_{s_i})P_{s_i,k_i}, \qquad z_{src} = [z_{s_1}; z_{s_2}; \ldots; z_{s_m}].
$$

The target is represented in target PCA space:

$$
z_t = (y_t - \mu_t)P_{t,k_t}.
$$

A scikit-learn `PLSRegression` model maps source PCA scores to target PCA scores:

$$
\hat z_t = f_{PLS}(z_{src}).
$$

The model decodes predicted target latents through the fixed target PCA basis:

$$
\hat y_t = \mu_t + \hat z_t P_{t,k_t}^{T}.
$$

In [ ]:
show_yaml_default("CrossModal_PCA_PLS")
show_sbatch_refs("CrossModal_PCA_PLS")

pca_pls_model = build_closed_form_model("CrossModal_PCA_PLS", data_sim.base)
pca_pls_run = evaluate_closed_form_model("CrossModal_PCA_PLS", pca_pls_model)
inspect_one_batch(pca_pls_model, "CrossModal_PCA_PLS one-batch check")
print("PLS rotations shape:", tuple(pca_pls_model.x_rotations_.shape))
print("target loadings shape:", tuple(pca_pls_model.target_loadings_k.shape))
pd.Series(pca_pls_run["test_metrics"]["base_metrics"]).sort_index()


## Closed-form local summary

This summary compares the local default-config smoke tests run on the shared data context. For production sweeps, use the referenced sbatch scripts rather than editing notebook parameters by hand.

In [ ]:
closed_form_summary = pd.DataFrame({
    "CrossModalPCA": pd.Series(pca_run["test_metrics"]["base_metrics"]),
    "CrossModal_PLS_SVD": pd.Series(pls_svd_run["test_metrics"]["base_metrics"]),
    "CrossModal_PCA_PLS": pd.Series(pca_pls_run["test_metrics"]["base_metrics"]),
}).T
closed_form_summary


In [ ]:
# ===================== STEP 1: SC -> SC ORACLE (ceiling) =====================
# Self-contained: IGNORES the SOURCE/TARGET toggle at the top. Builds its own
# SC->SC context and runs CrossModalPCA = a PCA round-trip of SC. Its
# demeaned_pearson is the CEILING (most individual SC structure recoverable).
# Compare FC->SC's PCA_PLS demeaned (~0.137) against this number.
# (Requires the first setup cell to have been run, for Sim/build_model/etc.)
import pandas as _pd

_oracle_sim = Sim(
    model_name="CrossModalPCA",
    config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
    source="SC", target="SC",
    parcellation=PARCELLATION,
    shuffle_seed=SHUFFLE_SEED,
    data_load_mode=DATA_LOAD_MODE,
)
_ocfg = get_default_config("CrossModalPCA", path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]))
_ocfg.setdefault("data", {})
_ocfg["data"].update({"source": "SC", "target": "SC", "parcellation": PARCELLATION,
                      "shuffle_seed": SHUFFLE_SEED, "data_load_mode": DATA_LOAD_MODE})
_ocfg.setdefault("model", {})
_ocfg["model"]["device"] = "cpu"
_ocfg = resolve_source_dependent_config(_ocfg)
_omcfg = deepcopy(_ocfg["model"]); _omcfg.pop("name", None)
_oracle_model = build_model(_oracle_sim.base, "CrossModalPCA", _omcfg).eval()
_oracle_out = _oracle_sim._evaluate_model(
    _oracle_model, mode="dev", model_name="CrossModalPCA_SC_SC_oracle",
    base=_oracle_sim.base, train_loader=_oracle_sim.train_loader,
    val_loader=_oracle_sim.val_loader, test_loader=_oracle_sim.test_loader,
)
print("=== SC -> SC self-reconstruction ORACLE (ceiling) ===")
print(_pd.Series(_oracle_out["test_metrics"]["base_metrics"]).sort_index())

In [ ]:
# ============= STEP 2a: 10-SEED ROBUSTNESS, 2-at-a-time (seeds 0..1) =============
# Split into 5 chunks of 2 seeds each (was 2 x 5) for tighter memory headroom on
# the 24 GB allocation: each cell runs only 2 Sims, and Python can fully reclaim
# memory between separate cell executions. Run 2a -> 2e in order.
# All chunks append to _seed_demeaned / _seed_top1 (initialized here).
import numpy as _np
import gc as _gc

# fresh accumulators (overwritten if 2a is re-run)
_seed_demeaned, _seed_top1 = [], []

def _run_seed_chunk(start, end):
    """Run FC->SC CrossModal_PCA_PLS for seeds in [start, end).
    Appends per-seed demeaned + top1 to the global accumulator lists."""
    for _s in range(start, end):
        _ss = Sim(
            model_name="CrossModal_PCA_PLS",
            config_path=str(CLOSED_FORM_MODELS["CrossModal_PCA_PLS"]["config"]),
            source="FC", target="SC",
            parcellation=PARCELLATION, shuffle_seed=_s, data_load_mode=DATA_LOAD_MODE,
        )
        _c = get_default_config("CrossModal_PCA_PLS", path=str(CLOSED_FORM_MODELS["CrossModal_PCA_PLS"]["config"]))
        _c.setdefault("data", {})
        _c["data"].update({"source": "FC", "target": "SC", "parcellation": PARCELLATION,
                           "shuffle_seed": _s, "data_load_mode": DATA_LOAD_MODE})
        _c.setdefault("model", {})
        _c["model"]["device"] = "cpu"
        _c = resolve_source_dependent_config(_c)
        _mc = deepcopy(_c["model"]); _mc.pop("name", None)
        _m = build_model(_ss.base, "CrossModal_PCA_PLS", _mc).eval()
        _o = _ss._evaluate_model(_m, mode="dev", model_name="CrossModal_PCA_PLS",
            base=_ss.base, train_loader=_ss.train_loader,
            val_loader=_ss.val_loader, test_loader=_ss.test_loader)
        _bm = _o["test_metrics"]["base_metrics"]
        _d, _t = float(_bm["demeaned_pearson"]), float(_bm["top1_acc"])
        _seed_demeaned.append(_d); _seed_top1.append(_t)
        print(f"seed {_s}: demeaned={_d:.4f}  top1={_t:.4f}", flush=True)
        del _o, _bm, _m, _mc, _c, _ss
        _gc.collect()
    print(f"\n{len(_seed_demeaned)}/10 seeds done. Running stats:")
    print(f"  demeaned = {_np.mean(_seed_demeaned):.4f} +/- {_np.std(_seed_demeaned):.4f}")
    print(f"  top1     = {_np.mean(_seed_top1):.4f} +/- {_np.std(_seed_top1):.4f}")

_run_seed_chunk(0, 2)

In [ ]:
# STEP 2b: seeds 2..3 (run AFTER 2a)
assert '_run_seed_chunk' in dir(), "Run STEP 2a first (it defines the helper and inits accumulators)."
_run_seed_chunk(2, 4)

In [ ]:
# STEP 2c: seeds 4..5
assert '_run_seed_chunk' in dir(), "Run STEP 2a first."
_run_seed_chunk(4, 6)

In [ ]:
# STEP 2d: seeds 6..7
assert '_run_seed_chunk' in dir(), "Run STEP 2a first."
_run_seed_chunk(6, 8)

In [ ]:
# STEP 2e: seeds 8..9 + FINAL 10-seed summary
assert '_run_seed_chunk' in dir(), "Run STEP 2a first."
_run_seed_chunk(8, 10)

import numpy as _np
print(f"\n=== FC->SC CrossModal_PCA_PLS, FINAL 10-seed summary ===")
print(f"  demeaned_pearson = {_np.mean(_seed_demeaned):.4f} +/- {_np.std(_seed_demeaned):.4f}")
print(f"  top1_acc         = {_np.mean(_seed_top1):.4f} +/- {_np.std(_seed_top1):.4f}")
print(f"  per-seed demeaned: {[f'{x:.4f}' for x in _seed_demeaned]}")
print(f"  per-seed top1:     {[f'{x:.4f}' for x in _seed_top1]}")

In [ ]:
# ============= HELPER: FULL 6-METRIC PANEL =============
# Single re-usable evaluator for all subsequent steps. Returns the same
# 6 metrics the production evaluator writes into `test_metrics["base_metrics"]`:
#   mse, r2, pearson, demeaned_pearson, top1_acc, avg_rank
#
# Used to retrofit Steps 3 / 3.5 / 3.6 / 5.1 with the full panel (not just
# demeaned_pearson) so identifiability metrics (top1_acc, avg_rank) and
# variance metrics (mse, r2, raw pearson) corroborate the demeaned-r story
# across the asymmetry and anatomy-control experiments.
#
# FIX (2026-05-26): demeaned_pearson now uses the project convention
# (cosine of (y - train_mean) vectors per subject, matching
# compute_demeaned_pearson_r), NOT row-Pearson on demeaned data. The two
# agree when residual row means are ~0 (Steps 3.5/3.6 residuals) but
# diverge by ~6% on raw predictions (Step 3, raw sanity checks). Now we
# match the project's published demeaned-r exactly.
import numpy as _np
from models.eval.metrics import (
    compute_corr_matrix,
    compute_basic_regression_metrics,
)


def _full_panel_eval(y_pred, y_true, target_train_mean_vec):
    """Compute all 6 metrics for (n_subjects, n_features) numpy preds & targets.

    Args
    ----
    y_pred              : (N, F) array-like
    y_true              : (N, F) array-like
    target_train_mean_vec : (F,) array-like  -- train mean of targets (for demeaning)

    Returns dict with keys: mse, r2, pearson, avg_rank, top1_acc, demeaned_pearson.

    Conventions
    -----------
    - mse, r2, pearson, top1_acc, avg_rank: from `compute_basic_regression_metrics`
      against the raw corr matrix (row-wise Pearson). Matches the project.
    - demeaned_pearson: per-subject cosine similarity of (y - train_mean) vectors
      (NOT row-Pearson on demeaned data). Matches `compute_demeaned_pearson_r`
      exactly, which is the project's published demeaned-r convention.
    """
    yp = _np.asarray(y_pred, dtype=_np.float32)
    yt = _np.asarray(y_true, dtype=_np.float32)
    mu = _np.asarray(target_train_mean_vec, dtype=_np.float32)

    # Raw row-Pearson corr matrix -> pearson, top1_acc, avg_rank, (mse, r2 are direct).
    cc_raw = compute_corr_matrix(yt, yp)
    panel = compute_basic_regression_metrics(
        yp, yt, corr_matrix=cc_raw, corr_matrix_demeaned=None
    )

    # Demeaned-r: cosine similarity of (y - mu) vectors, mean over subjects.
    # This is exactly compute_demeaned_pearson_r in numpy form.
    yp_dm = yp - mu
    yt_dm = yt - mu
    num   = (yp_dm * yt_dm).sum(axis=1)
    den_p = _np.sqrt((yp_dm ** 2).sum(axis=1))
    den_t = _np.sqrt((yt_dm ** 2).sum(axis=1))
    panel["demeaned_pearson"] = float((num / (den_p * den_t + 1e-10)).mean())
    return panel


def _fmt_panel(panel, label="", n_label_width=42):
    """One-line formatter for a panel dict (mse, r2, pearson, demeaned, top1, avg_rank)."""
    return (f"  {label:<{n_label_width}}"
            f"mse={panel['mse']:.5f}  r2={panel['r2']:+.4f}  "
            f"pearson={panel['pearson']:.4f}  demeaned={panel['demeaned_pearson']:.4f}  "
            f"top1={panel['top1_acc']:.4f}  avg_rank={panel['avg_rank']:.4f}")


print("_full_panel_eval and _fmt_panel are now defined "
      "(demeaned_pearson uses project's cosine-of-(y-mu) convention).")


In [25]:
# ============= STEP 3: BRAIN-VOLUME-ONLY BASELINE (confound check) =============
# Is FC->SC's 0.134 demeaned-r predicting individual *connectivity*, or just
# individual *brain anatomy* (size/volume)?
#
# Method: per-edge multilinear regression  SC_edge ~ FS_volume_features (16-dim)
# fit on TRAIN subjects, evaluated on TEST. We use the project's exact demeaned
# eval (subtract sc_train_avg, Pearson per subject, average) so this is
# apples-to-apples with FC->SC's 0.1341.
#
# If brain-vol-only reaches ~0.13  -> FC->SC is essentially just brain anatomy.
# If brain-vol-only is much lower  -> FC->SC has real individual connectivity above anatomy.
import numpy as _np
import torch as _torch
from sklearn.linear_model import LinearRegression
from models.eval.metrics import compute_demeaned_pearson_r

# Fresh FC->SC Sim -- only need base for raw SC + FS volumes; model_name doesn't matter
_s3_sim = Sim(model_name="CrossModalPCA",
              config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
              source="FC", target="SC",
              parcellation=PARCELLATION, shuffle_seed=0,
              data_load_mode=DATA_LOAD_MODE)
_base = _s3_sim.base

_train_idx = _base.trainvaltest_partition_indices["train"]
_test_idx  = _base.trainvaltest_partition_indices["test"]

# Inputs: FreeSurfer brain-volume features (z-scored on train, n_subjects x ~16)
_X_train = _base.fs_volumes_z[_train_idx]
_X_test  = _base.fs_volumes_z[_test_idx]
# Targets: raw SC edge vectors (log1p-transformed, matching what PCA_PLS sees)
_Y_train = _np.asarray(_base.sc_upper_triangles[_train_idx], dtype=_np.float32)
_Y_test  = _np.asarray(_base.sc_upper_triangles[_test_idx],  dtype=_np.float32)

print(f"X_train {_X_train.shape}, Y_train {_Y_train.shape}, X_test {_X_test.shape}, Y_test {_Y_test.shape}")
print(f"FS volume cols ({len(_base.fs_volume_columns)}): {_base.fs_volume_columns}")

# Per-edge multilinear regression: predict each of 64,620 SC edges from 16 brain-volume features.
# sklearn handles multi-output OLS efficiently in one call.
_reg = LinearRegression().fit(_X_train, _Y_train)
_Y_pred_test = _reg.predict(_X_test).astype(_np.float32)

# Use the project's exact demeaned_pearson for apples-to-apples comparison.
# (compute_demeaned_pearson_r already returns a Python float, no .item() needed.)
_target_train_mean = _torch.tensor(_base.sc_train_avg, dtype=_torch.float32)
_demeaned_r = float(compute_demeaned_pearson_r(
    _torch.tensor(_Y_pred_test, dtype=_torch.float32),
    _torch.tensor(_Y_test,      dtype=_torch.float32),
    _target_train_mean,
))

# Raw pearson per subject, averaged (for context vs the 0.91 FC->SC raw)
_pc = _Y_pred_test - _Y_pred_test.mean(axis=1, keepdims=True)
_tc = _Y_test      - _Y_test.mean(axis=1, keepdims=True)
_raw_r = float(_np.mean((_pc * _tc).sum(axis=1) /
                        (_np.sqrt((_pc**2).sum(axis=1)) * _np.sqrt((_tc**2).sum(axis=1)) + 1e-12)))

print(f"\n=== Brain-volume-only baseline (16 FS volume features -> 64620 SC edges) ===")
print(f"  raw pearson      = {_raw_r:.4f}")
print(f"  demeaned_pearson = {_demeaned_r:.4f}")
print(f"\nCompare:")
print(f"  FC->SC PCA_PLS demeaned (10-seed) = 0.1341 +/- 0.0054")
print(f"  SC->FC PCA_PLS demeaned (10-seed) = 0.0904 +/- 0.0101")
print(f"  SC->SC oracle (ceiling)            = 0.6472")
print(f"\nInterpretation:")
print(f"  If brain-vol-only ~= 0.134  -> FC->SC = just predicting brain anatomy (bad)")
print(f"  If brain-vol-only << 0.134  -> FC->SC captures individual connectivity above anatomy (good)")

# ---------- FULL 6-METRIC PANEL (retrofit, 2026-05-26) ----------
# The cell above only reported raw pearson and demeaned_pearson.
# This block re-evaluates the SAME prediction (_Y_pred_test) with the
# identifiability triad (top1_acc, avg_rank) and variance metrics (mse, r2).
# Key question: does brain-vol-only beat FC->SC on identifiability too,
# or only on demeaned_pearson?
_panel_step3 = _full_panel_eval(
    y_pred=_Y_pred_test,
    y_true=_Y_test,
    target_train_mean_vec=_base.sc_train_avg,
)
print("\n--- STEP 3 FULL PANEL: brain-vol -> SC (test set, n={}) ---".format(_Y_test.shape[0]))
print(_fmt_panel(_panel_step3, "brain-vol -> SC"))
print()
print("Compare on identifiability:")
print("  FC->SC PCA_PLS (project, seed 0)  : top1_acc=0.1538  avg_rank=0.8641   demeaned=0.1370")
print("  SC->FC PCA_PLS (project, seed 0)  : top1_acc=0.0513  avg_rank=0.7044   demeaned=0.0855")
print("  brain-vol -> SC (this cell)       : top1_acc={top1:.4f}  avg_rank={ar:.4f}   demeaned={dm:.4f}".format(
    top1=_panel_step3['top1_acc'], ar=_panel_step3['avg_rank'], dm=_panel_step3['demeaned_pearson']))
print()
print("Reframing test:")
print("  If brain-vol top1 ~= 0.154 -> anatomy ALSO wins identifiability -> anatomy story stands")
print("  If brain-vol top1  < 0.154 -> anatomy wins demeaned-r but FC wins identifiability")
print("                              -> the two metrics disagree, FC carries individuality anatomy misses")


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3023: RuntimeWarning: invalid value encountered in divide
  c /= stddev[:, None]
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/numpy/lib/_function_base_impl.py:3024: RuntimeWarning: invalid value encountered in divide
  c /= stddev[None, :]


X_train (683, 16), Y_train (683, 64620), X_test (195, 16), Y_test (195, 64620)
FS volume cols (16): ['FS_IntraCranial_Vol', 'FS_BrainSeg_Vol', 'FS_BrainSeg_Vol_No_Vent', 'FS_BrainSeg_Vol_No_Vent_Surf', 'FS_LCort_GM_Vol', 'FS_RCort_GM_Vol', 'FS_TotCort_GM_Vol', 'FS_SubCort_GM_Vol', 'FS_Total_GM_Vol', 'FS_SupraTentorial_Vol', 'FS_L_WM_Vol', 'FS_R_WM_Vol', 'FS_Tot_WM_Vol', 'FS_Mask_Vol', 'FS_BrainSegVol_eTIV_Ratio', 'FS_MaskVol_eTIV_Ratio']

=== Brain-volume-only baseline (16 FS volume features -> 64620 SC edges) ===
  raw pearson      = 0.9155
  demeaned_pearson = 0.1670

Compare:
  FC->SC PCA_PLS demeaned (10-seed) = 0.1341 +/- 0.0054
  SC->FC PCA_PLS demeaned (10-seed) = 0.0904 +/- 0.0101
  SC->SC oracle (ceiling)            = 0.6472

Interpretation:
  If brain-vol-only ~= 0.134  -> FC->SC = just predicting brain anatomy (bad)
  If brain-vol-only << 0.134  -> FC->SC captures individual connectivity above anatomy (good)

--- STEP 3 FULL PANEL: brain-vol -> SC (test set, n=195) ---
  bra

In [ ]:
# ============= STEP 3.5: Does FC predict SC ABOVE brain anatomy? =============
# Step 3 showed brain-vol-only -> SC gets demeaned 0.167, ABOVE FC->SC's 0.134.
# This cell tests: does FC predict SC variance that is ORTHOGONAL to brain anatomy?
#
# Method:
#   1. brain-vol OLS predicts SC on train + test.
#   2. SC_residual = SC - brain_vol_prediction (the part of SC anatomy can't explain).
#   3. FC -> SC_residual via PCA + sklearn PLSRegression (mirrors CrossModal_PCA_PLS).
#   4. Sanity: same manual pipeline FC -> SC (raw) -- should reproduce ~0.13.
#
# Interpretation of FC -> SC_residual demeaned-r:
#   ~0          : FC adds nothing above anatomy -> FC->SC story collapses to "FC predicts brain size"
#   ~0.05-0.10  : FC adds modest connectivity signal above anatomy
#   ~0.13       : FC and anatomy are orthogonal -> combined would beat both alone
import numpy as _np
import torch as _torch
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from models.eval.metrics import compute_demeaned_pearson_r

# Rebuild Sim if Step 3's _base isn't in scope (kernel may have been restarted)
if '_base' not in dir():
    _s35_sim = Sim(model_name="CrossModalPCA",
                   config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
                   source="FC", target="SC",
                   parcellation=PARCELLATION, shuffle_seed=0,
                   data_load_mode=DATA_LOAD_MODE)
    _base = _s35_sim.base

_train_idx = _base.trainvaltest_partition_indices["train"]
_test_idx  = _base.trainvaltest_partition_indices["test"]

# Pull raw arrays
_X_brainvol_train = _base.fs_volumes_z[_train_idx]
_X_brainvol_test  = _base.fs_volumes_z[_test_idx]
_SC_train = _np.asarray(_base.sc_upper_triangles[_train_idx], dtype=_np.float32)
_SC_test  = _np.asarray(_base.sc_upper_triangles[_test_idx],  dtype=_np.float32)
_FC_train = _np.asarray(_base.fc_upper_triangles[_train_idx], dtype=_np.float32)
_FC_test  = _np.asarray(_base.fc_upper_triangles[_test_idx],  dtype=_np.float32)
print(f"shapes: SC_train {_SC_train.shape}, FC_train {_FC_train.shape}, X_vol_train {_X_brainvol_train.shape}")

# --- 1. brain-vol -> SC prediction (per Step 3) ---
print("Fitting brain-vol OLS...", flush=True)
_bvreg = LinearRegression().fit(_X_brainvol_train, _SC_train)
_SC_brainvol_train = _bvreg.predict(_X_brainvol_train).astype(_np.float32)
_SC_brainvol_test  = _bvreg.predict(_X_brainvol_test ).astype(_np.float32)

# --- 2. SC residuals (the part anatomy can't explain) ---
_SC_resid_train = (_SC_train - _SC_brainvol_train).astype(_np.float32)
_SC_resid_test  = (_SC_test  - _SC_brainvol_test ).astype(_np.float32)
print(f"residual train mean (should be ~0): {_SC_resid_train.mean():.2e}, std: {_SC_resid_train.std():.4f}")

# --- 3. Generic FC -> Y pipeline (mirrors CrossModal_PCA_PLS: PCA src + PCA tgt + PLS latent) ---
N_PCA_SRC = 256   # match HCP_Base.num_pca_components_fc default
N_PCA_TGT = 256   # match HCP_Base.num_pca_components_sc default
N_PLS     = 64    # PLS latent dim (reasonable; CrossModal_PCA_PLS uses similar)

def _fc_to_target(Y_train, Y_test, k_src=N_PCA_SRC, k_tgt=N_PCA_TGT, k_pls=N_PLS):
    pca_src = PCA(n_components=k_src, random_state=0).fit(_FC_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_fc_train = pca_src.transform(_FC_train)
    Z_fc_test  = pca_src.transform(_FC_test)
    Z_y_train  = pca_tgt.transform(Y_train)
    pls = PLSRegression(n_components=k_pls, scale=True, max_iter=2000).fit(Z_fc_train, Z_y_train)  # max_iter=2000 (was default 500, hit convergence warning) -- matches Step 5.1 A
    Z_y_test_pred = pls.predict(Z_fc_test)
    Y_test_pred   = pca_tgt.inverse_transform(Z_y_test_pred).astype(_np.float32)
    return Y_test_pred

def _demeaned_eval(Y_pred, Y_true, Y_train_mean_vec):
    return float(compute_demeaned_pearson_r(
        _torch.tensor(Y_pred, dtype=_torch.float32),
        _torch.tensor(Y_true, dtype=_torch.float32),
        _torch.tensor(Y_train_mean_vec, dtype=_torch.float32),
    ))

# --- 3a. SANITY CHECK: manual FC -> SC (raw, no residualization) ---
# Should reproduce ~0.134 if this manual pipeline matches project PCA_PLS reasonably.
print("Running sanity check: manual FC -> SC (raw)...", flush=True)
_SC_test_pred_raw = _fc_to_target(_SC_train, _SC_test)
_d_sanity = _demeaned_eval(_SC_test_pred_raw, _SC_test, _SC_train.mean(axis=0))

# --- 3b. THE ACTUAL TEST: manual FC -> SC_residual ---
print("Running main test: manual FC -> SC_residual...", flush=True)
_SC_resid_test_pred = _fc_to_target(_SC_resid_train, _SC_resid_test)
_d_resid = _demeaned_eval(_SC_resid_test_pred, _SC_resid_test, _SC_resid_train.mean(axis=0))

print(f"\n=== STEP 3.5: FC predicting SC RESIDUALS (anatomy removed) ===")
print(f"  (sanity)  FC -> SC (raw, manual pipeline)      demeaned = {_d_sanity:.4f}")
print(f"            project PCA_PLS (10-seed)                     ~ 0.1341")
print(f"            -> close means my pipeline is calibrated\n")
print(f"  (main)    FC -> SC_residual (anatomy removed)  demeaned = {_d_resid:.4f}")
print(f"\nContext:")
print(f"  brain-vol-only -> SC          = 0.1670")
print(f"  FC -> SC (project, 10-seed)   = 0.1341 +/- 0.0054")
print(f"  FC -> SC_residual (this cell) = {_d_resid:.4f}")
print(f"\nInterpretation:")
print(f"  ~0         : FC adds nothing above anatomy -> FC->SC = brain-size proxy")
print(f"  ~0.05-0.10 : FC adds modest connectivity signal above anatomy")
print(f"  ~0.13      : FC and anatomy are orthogonal -> combined would beat both")

# ---------- FULL 6-METRIC PANEL (retrofit, 2026-05-26) ----------
# Re-evaluate the three predictions this cell made:
#   (a) brain-vol -> SC                (anatomy alone; baseline)
#   (b) FC -> SC raw (manual sanity)   (matches the project's 0.134 pipeline)
#   (c) FC -> SC_residual              (main test: FC orthogonal to anatomy)
# The original cell only reported demeaned_pearson. We add the identifiability
# triad (top1_acc, avg_rank) + variance metrics (mse, r2, raw pearson).
#
# Why on residual targets too:
#   top1_acc(residual) answers "can FC ID subjects from their *connectivity-residual*
#   SC?" — i.e., is there enough FC-specific subject signal in SC-orthogonal-to-anatomy
#   to fingerprint subjects. A non-trivial result here would be substantively new.
_panel_35_bv  = _full_panel_eval(
    y_pred=_SC_brainvol_test, y_true=_SC_test,
    target_train_mean_vec=_SC_train.mean(axis=0),
)
_panel_35_raw = _full_panel_eval(
    y_pred=_SC_test_pred_raw, y_true=_SC_test,
    target_train_mean_vec=_SC_train.mean(axis=0),
)
_panel_35_res = _full_panel_eval(
    y_pred=_SC_resid_test_pred, y_true=_SC_resid_test,
    target_train_mean_vec=_SC_resid_train.mean(axis=0),
)
print("\n--- STEP 3.5 FULL PANEL: target = SC (n={} test subjects) ---".format(_SC_test.shape[0]))
print(_fmt_panel(_panel_35_bv,  "brain-vol -> SC                       "))
print(_fmt_panel(_panel_35_raw, "FC -> SC (raw, manual sanity)         "))
print(_fmt_panel(_panel_35_res, "FC -> SC_residual (anatomy removed)   "))
print()
print("Identifiability triad summary:")
print("                                       top1_acc   avg_rank   demeaned_r")
print(f"  brain-vol -> SC                    : {_panel_35_bv['top1_acc']:.4f}     {_panel_35_bv['avg_rank']:.4f}     {_panel_35_bv['demeaned_pearson']:.4f}")
print(f"  FC -> SC (raw)                     : {_panel_35_raw['top1_acc']:.4f}     {_panel_35_raw['avg_rank']:.4f}     {_panel_35_raw['demeaned_pearson']:.4f}")
print(f"  FC -> SC_residual (anatomy-out)    : {_panel_35_res['top1_acc']:.4f}     {_panel_35_res['avg_rank']:.4f}     {_panel_35_res['demeaned_pearson']:.4f}")


In [ ]:
# ============= STEP 3.6: SYMMETRIC -- SC -> FC after removing anatomy =============
# Mirror of Step 3.5 for the SC->FC direction. The asymmetry test:
#
#   Raw:       FC->SC = 0.134,  SC->FC = 0.090   (+49% lift, anatomy-CONFOUNDED)
#   Residual:  FC->SC_resid = 0.0778, SC->FC_resid = ??   (anatomy-controlled, the fair test)
#
# Method (mirror of 3.5 with directions flipped):
#   1. brain-vol OLS predicts FC on train + test.
#   2. FC_residual = FC - brain_vol_prediction.
#   3. SC -> FC_residual via PCA + sklearn PLSRegression.
#   4. Sanity: same pipeline SC -> FC (raw) -- should reproduce ~0.09.
import numpy as _np
import torch as _torch
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import LinearRegression
from models.eval.metrics import compute_demeaned_pearson_r

# Reuse _base + arrays if Step 3.5 already loaded them
if '_base' not in dir():
    _s36_sim = Sim(model_name="CrossModalPCA",
                   config_path=str(CLOSED_FORM_MODELS["CrossModalPCA"]["config"]),
                   source="SC", target="FC",
                   parcellation=PARCELLATION, shuffle_seed=0,
                   data_load_mode=DATA_LOAD_MODE)
    _base = _s36_sim.base

_train_idx = _base.trainvaltest_partition_indices["train"]
_test_idx  = _base.trainvaltest_partition_indices["test"]

_X_vol_train = _base.fs_volumes_z[_train_idx]
_X_vol_test  = _base.fs_volumes_z[_test_idx]
_SC_train = _np.asarray(_base.sc_upper_triangles[_train_idx], dtype=_np.float32)
_SC_test  = _np.asarray(_base.sc_upper_triangles[_test_idx],  dtype=_np.float32)
_FC_train = _np.asarray(_base.fc_upper_triangles[_train_idx], dtype=_np.float32)
_FC_test  = _np.asarray(_base.fc_upper_triangles[_test_idx],  dtype=_np.float32)

# --- 1. brain-vol -> FC prediction (symmetric of Step 3) ---
print("Fitting brain-vol -> FC OLS...", flush=True)
_bvreg_fc = LinearRegression().fit(_X_vol_train, _FC_train)
_FC_bv_train = _bvreg_fc.predict(_X_vol_train).astype(_np.float32)
_FC_bv_test  = _bvreg_fc.predict(_X_vol_test ).astype(_np.float32)

# brain-vol -> FC demeaned-r (analog of brain-vol -> SC = 0.167)
_d_bv_fc = float(compute_demeaned_pearson_r(
    _torch.tensor(_FC_bv_test, dtype=_torch.float32),
    _torch.tensor(_FC_test, dtype=_torch.float32),
    _torch.tensor(_FC_train.mean(axis=0), dtype=_torch.float32),
))
print(f"brain-vol -> FC demeaned = {_d_bv_fc:.4f}  (compare to brain-vol -> SC = 0.1670)")

# --- 2. FC residuals (the part of FC anatomy can't explain) ---
_FC_resid_train = (_FC_train - _FC_bv_train).astype(_np.float32)
_FC_resid_test  = (_FC_test  - _FC_bv_test ).astype(_np.float32)
print(f"FC residual train mean: {_FC_resid_train.mean():.2e}, std: {_FC_resid_train.std():.4f}")

# --- 3. Generic SC -> Y pipeline (mirror of FC->Y from 3.5) ---
N_PCA_SRC = 256
N_PCA_TGT = 256
N_PLS     = 64

def _sc_to_target(Y_train, Y_test, k_src=N_PCA_SRC, k_tgt=N_PCA_TGT, k_pls=N_PLS):
    pca_src = PCA(n_components=k_src, random_state=0).fit(_SC_train)
    pca_tgt = PCA(n_components=k_tgt, random_state=0).fit(Y_train)
    Z_sc_train = pca_src.transform(_SC_train)
    Z_sc_test  = pca_src.transform(_SC_test)
    Z_y_train  = pca_tgt.transform(Y_train)
    pls = PLSRegression(n_components=k_pls, scale=True, max_iter=2000).fit(Z_sc_train, Z_y_train)  # max_iter=2000 (was default 500, hit convergence warning) -- matches Step 5.1 A
    Z_y_test_pred = pls.predict(Z_sc_test)
    return pca_tgt.inverse_transform(Z_y_test_pred).astype(_np.float32)

def _demeaned_eval(Y_pred, Y_true, Y_train_mean_vec):
    return float(compute_demeaned_pearson_r(
        _torch.tensor(Y_pred, dtype=_torch.float32),
        _torch.tensor(Y_true, dtype=_torch.float32),
        _torch.tensor(Y_train_mean_vec, dtype=_torch.float32),
    ))

# --- 3a. SANITY: manual SC -> FC (raw) -- should reproduce ~0.09 ---
print("Running sanity: manual SC -> FC (raw)...", flush=True)
_FC_test_pred_raw = _sc_to_target(_FC_train, _FC_test)
_d_sc_fc_raw = _demeaned_eval(_FC_test_pred_raw, _FC_test, _FC_train.mean(axis=0))

# --- 3b. MAIN: SC -> FC_residual ---
print("Running main test: manual SC -> FC_residual...", flush=True)
_FC_resid_test_pred = _sc_to_target(_FC_resid_train, _FC_resid_test)
_d_sc_fc_resid = _demeaned_eval(_FC_resid_test_pred, _FC_resid_test, _FC_resid_train.mean(axis=0))

print(f"\n=== STEP 3.6: SYMMETRIC ANATOMY-CONTROLLED COMPARISON ===")
print(f"  brain-vol -> SC demeaned = 0.1670  (Step 3)")
print(f"  brain-vol -> FC demeaned = {_d_bv_fc:.4f}  (this cell)")
print()
print(f"  (sanity) SC -> FC raw, manual pipeline = {_d_sc_fc_raw:.4f}")
print(f"           project PCA_PLS (10-seed)     ~ 0.0904")
print()
print(f"  ====== THE ASYMMETRY TEST (anatomy-controlled) ======")
print(f"  FC -> SC_residual  (Step 3.5)  = 0.0778")
print(f"  SC -> FC_residual  (this cell) = {_d_sc_fc_resid:.4f}")
print()
print(f"Interpretation:")
print(f"  If SC -> FC_residual << 0.0778  -> asymmetry SURVIVES anatomy control")
print(f"                                  -> FC->SC has more connectivity signal even stripping anatomy (pivot validated)")
print(f"  If SC -> FC_residual ~= 0.0778  -> asymmetry DISAPPEARS in anatomy-controlled space")
print(f"                                  -> raw +49% lift was mostly anatomy; both directions modestly above 0")
print(f"  If SC -> FC_residual >  0.0778  -> reverse asymmetry (unexpected)")

# ---------- FULL 6-METRIC PANEL (retrofit, 2026-05-26) ----------
# Mirror of Step 3.5's full-panel retrofit, FC side:
#   (a) brain-vol -> FC               (anatomy alone, FC side)
#   (b) SC -> FC raw (manual sanity)
#   (c) SC -> FC_residual             (main: SC orthogonal to anatomy)
# Together with Step 3.5 this gives the directional asymmetry on all 6 metrics.
_panel_36_bv  = _full_panel_eval(
    y_pred=_FC_bv_test, y_true=_FC_test,
    target_train_mean_vec=_FC_train.mean(axis=0),
)
_panel_36_raw = _full_panel_eval(
    y_pred=_FC_test_pred_raw, y_true=_FC_test,
    target_train_mean_vec=_FC_train.mean(axis=0),
)
_panel_36_res = _full_panel_eval(
    y_pred=_FC_resid_test_pred, y_true=_FC_resid_test,
    target_train_mean_vec=_FC_resid_train.mean(axis=0),
)
print("\n--- STEP 3.6 FULL PANEL: target = FC (n={} test subjects) ---".format(_FC_test.shape[0]))
print(_fmt_panel(_panel_36_bv,  "brain-vol -> FC                       "))
print(_fmt_panel(_panel_36_raw, "SC -> FC (raw, manual sanity)         "))
print(_fmt_panel(_panel_36_res, "SC -> FC_residual (anatomy removed)   "))
print()
print("Identifiability triad summary (FC target):")
print("                                       top1_acc   avg_rank   demeaned_r")
print(f"  brain-vol -> FC                    : {_panel_36_bv['top1_acc']:.4f}     {_panel_36_bv['avg_rank']:.4f}     {_panel_36_bv['demeaned_pearson']:.4f}")
print(f"  SC -> FC (raw)                     : {_panel_36_raw['top1_acc']:.4f}     {_panel_36_raw['avg_rank']:.4f}     {_panel_36_raw['demeaned_pearson']:.4f}")
print(f"  SC -> FC_residual (anatomy-out)    : {_panel_36_res['top1_acc']:.4f}     {_panel_36_res['avg_rank']:.4f}     {_panel_36_res['demeaned_pearson']:.4f}")
print()
print("=== CROSS-DIRECTION ASYMMETRY (anatomy-controlled, identifiability triad) ===")
print("                       FC->SC_resid   SC->FC_resid   ratio (FC->SC / SC->FC)")
print(f"  top1_acc           : {_panel_35_res['top1_acc']:.4f}        {_panel_36_res['top1_acc']:.4f}        {_panel_35_res['top1_acc']/max(_panel_36_res['top1_acc'],1e-12):.3f}x")
print(f"  avg_rank           : {_panel_35_res['avg_rank']:.4f}        {_panel_36_res['avg_rank']:.4f}        {_panel_35_res['avg_rank']/max(_panel_36_res['avg_rank'],1e-12):.3f}x")
print(f"  demeaned_pearson   : {_panel_35_res['demeaned_pearson']:.4f}        {_panel_36_res['demeaned_pearson']:.4f}        {_panel_35_res['demeaned_pearson']/max(_panel_36_res['demeaned_pearson'],1e-12):.3f}x")
print()
print("Reading:")
print("  If all 3 ratios > 1 and similar magnitude (~1.3-1.5x) -> asymmetry is robust across metrics")
print("  If demeaned_r > 1 but top1/avg_rank ~ 1 (or < 1)      -> demeaned-r asymmetry is metric-specific")
print("  If demeaned_r and top1 disagree                       -> different aspects of asymmetry")


In [15]:
# ============= STEP 4.1: BOOTSTRAP CIs ON RESIDUAL DEMEANED-R =============
# Steps 3, 3.5, 3.6 gave point estimates. This gives proper 95% CIs by 
# resampling TEST subjects (with replacement) 1000x and recomputing demeaned-r.
#
# Captures TEST-SET sampling uncertainty (the main source for a deterministic 
# closed-form model on a fixed training fit). Does NOT capture train-fit uncertainty 
# (would need to refit OLS+PCA+PLS per bootstrap = expensive). For these residual 
# numbers, this is the cheap honest CI.
import numpy as _np

def _bootstrap_demeaned_r(Y_pred, Y_true, Y_train_mean_vec, n_iter=1000, seed=42):
    """Resample test subjects with replacement; recompute demeaned-r per draw."""
    rng = _np.random.default_rng(seed)
    n_subjects = Y_pred.shape[0]
    Y_pred_d = Y_pred - Y_train_mean_vec
    Y_true_d = Y_true - Y_train_mean_vec
    # per-subject Pearson (computed once, then resampled)
    num = (Y_pred_d * Y_true_d).sum(axis=1)
    den = _np.sqrt((Y_pred_d ** 2).sum(axis=1)) * _np.sqrt((Y_true_d ** 2).sum(axis=1)) + 1e-12
    r_per_subj = num / den  # (n_subjects,)
    out = _np.zeros(n_iter)
    for i in range(n_iter):
        idx = rng.choice(n_subjects, size=n_subjects, replace=True)
        out[i] = r_per_subj[idx].mean()
    return out

print("Bootstrapping (1000 iters)...")

boot_FC_SC_resid = _bootstrap_demeaned_r(
    _SC_resid_test_pred, _SC_resid_test, _SC_resid_train.mean(axis=0)
)
boot_SC_FC_resid = _bootstrap_demeaned_r(
    _FC_resid_test_pred, _FC_resid_test, _FC_resid_train.mean(axis=0)
)
boot_BV_SC = _bootstrap_demeaned_r(
    _SC_brainvol_test, _SC_test, _base.sc_train_avg
)
boot_BV_FC = _bootstrap_demeaned_r(
    _FC_bv_test, _FC_test, _FC_train.mean(axis=0)
)

def _summarize(name, b):
    lo, hi = _np.percentile(b, [2.5, 97.5])
    print(f"  {name:25s} mean = {_np.mean(b):.4f}   95% CI [{lo:.4f}, {hi:.4f}]   std = {_np.std(b):.4f}")

print(f"\n=== Bootstrap 95% CIs (1000 iters, test-subject resampling) ===")
_summarize("brain-vol -> SC",     boot_BV_SC)
_summarize("brain-vol -> FC",     boot_BV_FC)
_summarize("FC -> SC_residual",   boot_FC_SC_resid)
_summarize("SC -> FC_residual",   boot_SC_FC_resid)

# Asymmetry test in bootstrap space
diff = boot_FC_SC_resid - boot_SC_FC_resid
lo, hi = _np.percentile(diff, [2.5, 97.5])
p_pos = (diff > 0).mean()
print(f"\n=== Asymmetry: (FC->SC_resid) - (SC->FC_resid) ===")
print(f"  diff mean = {_np.mean(diff):.4f}   95% CI [{lo:.4f}, {hi:.4f}]")
print(f"  P(diff > 0) = {p_pos:.3f}  (fraction of bootstrap iters where FC->SC > SC->FC)")
print(f"\nIf the 95% CI on diff excludes 0, the asymmetry is statistically significant.")
print(f"P(diff>0) near 1.0 means the asymmetry direction is essentially certain.")

Bootstrapping (1000 iters)...

=== Bootstrap 95% CIs (1000 iters, test-subject resampling) ===
  brain-vol -> SC           mean = 0.1668   95% CI [0.1535, 0.1803]   std = 0.0068
  brain-vol -> FC           mean = 0.0468   95% CI [0.0318, 0.0596]   std = 0.0072
  FC -> SC_residual         mean = 0.0781   95% CI [0.0691, 0.0868]   std = 0.0045
  SC -> FC_residual         mean = 0.0561   95% CI [0.0387, 0.0738]   std = 0.0088

=== Asymmetry: (FC->SC_resid) - (SC->FC_resid) ===
  diff mean = 0.0220   95% CI [0.0061, 0.0375]
  P(diff > 0) = 0.996  (fraction of bootstrap iters where FC->SC > SC->FC)

If the 95% CI on diff excludes 0, the asymmetry is statistically significant.
P(diff>0) near 1.0 means the asymmetry direction is essentially certain.


In [16]:
# ============= STEP 4.2: PERMUTATION TEST FOR FC->SC_residual SIGNIFICANCE =============
# Null hypothesis H0: the FC predictions are random with respect to SC residuals.
# Under H0, shuffling FC predictions across test subjects should give a similar 
# demeaned-r distribution to the observed value.
#
# Simulate H0: permute predictions across subjects 1000x → empirical null distribution.
# Empirical p-value = P(null >= observed). Tests "is the connectivity signal 
# above what random pairings would give?"
#
# Tests both directions' anatomy-controlled signal for significance.
import numpy as _np

def _perm_demeaned_r(Y_pred_orig, Y_true, Y_train_mean_vec, n_iter=1000, seed=42):
    """Shuffle predictions across subjects, recompute demeaned-r each draw."""
    rng = _np.random.default_rng(seed)
    n_subjects = Y_pred_orig.shape[0]
    Y_true_d = Y_true - Y_train_mean_vec
    out = _np.zeros(n_iter)
    for i in range(n_iter):
        perm = rng.permutation(n_subjects)
        Y_pred_perm = Y_pred_orig[perm] - Y_train_mean_vec
        num = (Y_pred_perm * Y_true_d).sum(axis=1)
        den = _np.sqrt((Y_pred_perm ** 2).sum(axis=1)) * _np.sqrt((Y_true_d ** 2).sum(axis=1)) + 1e-12
        out[i] = (num / den).mean()
    return out

print("Permutation testing (1000 iters)...")

obs_FC_SC = 0.0778
obs_SC_FC = 0.0561

perm_FC_SC = _perm_demeaned_r(
    _SC_resid_test_pred, _SC_resid_test, _SC_resid_train.mean(axis=0)
)
perm_SC_FC = _perm_demeaned_r(
    _FC_resid_test_pred, _FC_resid_test, _FC_resid_train.mean(axis=0)
)

def _perm_report(name, perm_dist, observed):
    p_val = (perm_dist >= observed).mean()
    null_mean = perm_dist.mean()
    null_std  = perm_dist.std()
    null_q975 = _np.percentile(perm_dist, 97.5)
    z = (observed - null_mean) / (null_std + 1e-12)
    sig = "*** highly significant" if p_val < 0.001 else "** significant (p<.05)" if p_val < 0.05 else "n.s."
    print(f"  {name}:")
    print(f"    observed                = {observed:.4f}")
    print(f"    null mean +/- std       = {null_mean:.4f} +/- {null_std:.4f}")
    print(f"    null 97.5th pctile      = {null_q975:.4f}  (one-sided 0.025 critical)")
    print(f"    z-score                 = {z:.2f}")
    print(f"    permutation p-value     = {p_val:.4f}")
    print(f"    -> {sig}")

print("\n=== Permutation test results ===")
_perm_report("FC -> SC_residual", perm_FC_SC, obs_FC_SC)
print()
_perm_report("SC -> FC_residual", perm_SC_FC, obs_SC_FC)
print()
print("Tests whether each direction's anatomy-controlled prediction is significantly")
print("above what a shuffled (random) prediction would give. Low p -> real signal.")

Permutation testing (1000 iters)...

=== Permutation test results ===
  FC -> SC_residual:
    observed                = 0.0778
    null mean +/- std       = 0.0018 +/- 0.0042
    null 97.5th pctile      = 0.0096  (one-sided 0.025 critical)
    z-score                 = 18.12
    permutation p-value     = 0.0000
    -> *** highly significant

  SC -> FC_residual:
    observed                = 0.0561
    null mean +/- std       = 0.0017 +/- 0.0079
    null 97.5th pctile      = 0.0158  (one-sided 0.025 critical)
    z-score                 = 6.87
    permutation p-value     = 0.0000
    -> *** highly significant

Tests whether each direction's anatomy-controlled prediction is significantly
above what a shuffled (random) prediction would give. Low p -> real signal.


In [17]:
# ============= STEP 4.3: BAYESIAN VARIANCE DECOMPOSITION =============
# Per-component conjugate Bayesian linear regression on SC PCA latents.
#   z_SC_k = X_anatomy @ β_a + Z_FC_PCA @ β_f + ε
# Uses sklearn's BayesianRidge (analytical posterior via type-II MLE for hyperparams).
#
# Decomposes test-set variance per SC component into:
#   - R² anatomy-only       (what anatomy alone explains)
#   - R² FC-only            (what FC alone explains)
#   - R² anatomy + FC       (joint)
# Then (Cohen-style variance partition):
#   - Unique to anatomy     = R²_full - R²_FC
#   - Unique to FC          = R²_full - R²_anatomy
#   - Shared anatomy+FC     = R²_anatomy + R²_FC - R²_full
#
# "Unique to FC" is the proper Bayesian analog of FC->SC_residual: the SC variance
# FC explains that anatomy cannot.
import numpy as _np
from sklearn.decomposition import PCA
from sklearn.linear_model import BayesianRidge

K_PCA_SC_LATENT = 64    # SC reduced to 64 PCA components (interpretable, fast)
K_PCA_FC_LATENT = 256   # FC PCA features as predictors

print(f"Reducing SC -> {K_PCA_SC_LATENT} PCA components, FC -> {K_PCA_FC_LATENT} PCA features...")
pca_sc = PCA(n_components=K_PCA_SC_LATENT, random_state=0).fit(_SC_train)
pca_fc = PCA(n_components=K_PCA_FC_LATENT, random_state=0).fit(_FC_train)
Z_SC_train = pca_sc.transform(_SC_train)
Z_SC_test  = pca_sc.transform(_SC_test)
Z_FC_train = pca_fc.transform(_FC_train)
Z_FC_test  = pca_fc.transform(_FC_test)

n_anat = _X_brainvol_train.shape[1]
n_fc   = K_PCA_FC_LATENT
print(f"feature counts: anatomy={n_anat}, FC_PCA={n_fc}, total={n_anat+n_fc}")

X_anat_train, X_anat_test = _X_brainvol_train, _X_brainvol_test
X_fc_train,   X_fc_test   = Z_FC_train, Z_FC_test
X_full_train = _np.concatenate([X_anat_train, X_fc_train], axis=1)
X_full_test  = _np.concatenate([X_anat_test,  X_fc_test ], axis=1)

print(f"\nFitting Bayesian Ridge for {K_PCA_SC_LATENT} SC components x 3 models each ({3*K_PCA_SC_LATENT} fits)...")
r2_anat_per_k, r2_fc_per_k, r2_full_per_k = [], [], []
for k in range(K_PCA_SC_LATENT):
    y_train_k = Z_SC_train[:, k]
    y_test_k  = Z_SC_test[:, k]
    ss_total  = ((y_test_k - y_test_k.mean()) ** 2).sum()
    if ss_total < 1e-12:
        r2_anat_per_k.append(0.0); r2_fc_per_k.append(0.0); r2_full_per_k.append(0.0); continue
    m_anat = BayesianRidge(max_iter=300).fit(X_anat_train, y_train_k)
    m_fc   = BayesianRidge(max_iter=300).fit(X_fc_train,   y_train_k)
    m_full = BayesianRidge(max_iter=300).fit(X_full_train, y_train_k)
    pred_anat = m_anat.predict(X_anat_test)
    pred_fc   = m_fc.predict(X_fc_test)
    pred_full = m_full.predict(X_full_test)
    r2_anat = 1 - ((y_test_k - pred_anat) ** 2).sum() / ss_total
    r2_fc   = 1 - ((y_test_k - pred_fc)   ** 2).sum() / ss_total
    r2_full = 1 - ((y_test_k - pred_full) ** 2).sum() / ss_total
    r2_anat_per_k.append(max(0, r2_anat))
    r2_fc_per_k.append(  max(0, r2_fc))
    r2_full_per_k.append(max(0, r2_full))

r2_anat = _np.array(r2_anat_per_k)
r2_fc   = _np.array(r2_fc_per_k)
r2_full = _np.array(r2_full_per_k)
unique_anat = _np.maximum(0, r2_full - r2_fc)
unique_fc   = _np.maximum(0, r2_full - r2_anat)
shared      = _np.maximum(0, r2_anat + r2_fc - r2_full)

print(f"\n=== Bayesian variance decomposition (test set, {K_PCA_SC_LATENT} SC PCA components) ===")
print(f"  R² anatomy-only       mean={r2_anat.mean():.4f}  sum={r2_anat.sum():.4f}")
print(f"  R² FC-only            mean={r2_fc.mean():.4f}    sum={r2_fc.sum():.4f}")
print(f"  R² anatomy + FC       mean={r2_full.mean():.4f}  sum={r2_full.sum():.4f}")
print()
print(f"  Unique to anatomy     mean={unique_anat.mean():.4f}  sum={unique_anat.sum():.4f}")
print(f"  Unique to FC          mean={unique_fc.mean():.4f}    sum={unique_fc.sum():.4f}")
print(f"  Shared anatomy+FC     mean={shared.mean():.4f}       sum={shared.sum():.4f}")
print()
print(f"Implication:")
print(f"  - 'Unique to FC' > 0 means FC carries SC variance anatomy alone cannot capture.")
print(f"  - 'Shared' high means FC and anatomy predict largely OVERLAPPING SC variance.")
print(f"  - Top 5 SC components ranked by 'Unique to FC':")
for idx in _np.argsort(unique_fc)[::-1][:5]:
    print(f"    PC{idx:3d}: anat={r2_anat[idx]:.3f}  FC={r2_fc[idx]:.3f}  full={r2_full[idx]:.3f}  unique-FC={unique_fc[idx]:.3f}")

Reducing SC -> 64 PCA components, FC -> 256 PCA features...
feature counts: anatomy=16, FC_PCA=256, total=272

Fitting Bayesian Ridge for 64 SC components x 3 models each (192 fits)...

=== Bayesian variance decomposition (test set, 64 SC PCA components) ===
  R² anatomy-only       mean=0.0265  sum=1.6967
  R² FC-only            mean=0.0317    sum=2.0279
  R² anatomy + FC       mean=0.0398  sum=2.5453

  Unique to anatomy     mean=0.0091  sum=0.5803
  Unique to FC          mean=0.0205    sum=1.3124
  Shared anatomy+FC     mean=0.0193       sum=1.2368

Implication:
  - 'Unique to FC' > 0 means FC carries SC variance anatomy alone cannot capture.
  - 'Shared' high means FC and anatomy predict largely OVERLAPPING SC variance.
  - Top 5 SC components ranked by 'Unique to FC':
    PC  2: anat=0.000  FC=0.256  full=0.259  unique-FC=0.259
    PC  3: anat=0.000  FC=0.091  full=0.093  unique-FC=0.093
    PC 25: anat=0.000  FC=0.090  full=0.088  unique-FC=0.088
    PC  4: anat=0.000  FC=0.091  f

In [19]:
# ============= STEP 4.4: SYMMETRIC BAYESIAN VARIANCE DECOMPOSITION (TARGET = FC) =============
# Mirror of Step 4.3 with directions swapped: target is now FC, predictors are anatomy + SC PCA.
#   z_FC_k = X_anatomy @ beta_a + Z_SC_PCA @ beta_s + eps
#
# Completes the symmetric anatomy-vs-cross-modal decomposition. Compare to 4.3:
#   - 4.3 (target = SC): Unique-to-FC = 0.0205 (how much SC variance FC uniquely captures above anatomy)
#   - 4.4 (target = FC): Unique-to-SC = ???    (how much FC variance SC uniquely captures above anatomy)
# If Unique-to-FC (4.3) > Unique-to-SC (this cell), the FC->SC asymmetry holds in variance-partition
# space too -- consistent with the demeaned-r view (1.39x), bootstrap (P(diff>0)=0.996), and
# permutation (z=18 vs z=6.87).
import numpy as _np
from sklearn.decomposition import PCA
from sklearn.linear_model import BayesianRidge

K_PCA_FC_TGT = 64    # FC reduced to 64 PCA components (target)
K_PCA_SC_SRC = 256   # SC PCA features as predictors

print(f"Reducing FC -> {K_PCA_FC_TGT} PCA components (target), SC -> {K_PCA_SC_SRC} (predictor)...")
pca_fc_tgt = PCA(n_components=K_PCA_FC_TGT, random_state=0).fit(_FC_train)
pca_sc_src = PCA(n_components=K_PCA_SC_SRC, random_state=0).fit(_SC_train)
Z_FC_train_tgt = pca_fc_tgt.transform(_FC_train)
Z_FC_test_tgt  = pca_fc_tgt.transform(_FC_test)
Z_SC_train_src = pca_sc_src.transform(_SC_train)
Z_SC_test_src  = pca_sc_src.transform(_SC_test)

n_anat_s = _X_brainvol_train.shape[1]
n_sc_s   = K_PCA_SC_SRC
print(f"feature counts: anatomy={n_anat_s}, SC_PCA={n_sc_s}, total={n_anat_s+n_sc_s}")

X_anat_tr, X_anat_te = _X_brainvol_train, _X_brainvol_test
X_sc_tr,   X_sc_te   = Z_SC_train_src, Z_SC_test_src
X_full_tr_s = _np.concatenate([X_anat_tr, X_sc_tr], axis=1)
X_full_te_s = _np.concatenate([X_anat_te, X_sc_te], axis=1)

print(f"\nFitting Bayesian Ridge for {K_PCA_FC_TGT} FC components x 3 models each ({3*K_PCA_FC_TGT} fits)...")
r2_anat_sym, r2_sc_sym, r2_full_sym = [], [], []
for k in range(K_PCA_FC_TGT):
    y_tr = Z_FC_train_tgt[:, k]
    y_te = Z_FC_test_tgt[:, k]
    ss_total = ((y_te - y_te.mean()) ** 2).sum()
    if ss_total < 1e-12:
        r2_anat_sym.append(0.0); r2_sc_sym.append(0.0); r2_full_sym.append(0.0); continue
    m_anat = BayesianRidge(max_iter=300).fit(X_anat_tr,   y_tr)
    m_sc   = BayesianRidge(max_iter=300).fit(X_sc_tr,     y_tr)
    m_full = BayesianRidge(max_iter=300).fit(X_full_tr_s, y_tr)
    r2_a = 1 - ((y_te - m_anat.predict(X_anat_te))   ** 2).sum() / ss_total
    r2_s = 1 - ((y_te - m_sc.predict(X_sc_te))       ** 2).sum() / ss_total
    r2_f = 1 - ((y_te - m_full.predict(X_full_te_s)) ** 2).sum() / ss_total
    r2_anat_sym.append(max(0, r2_a))
    r2_sc_sym.append(  max(0, r2_s))
    r2_full_sym.append(max(0, r2_f))

r2_anat_sym = _np.array(r2_anat_sym)
r2_sc_sym   = _np.array(r2_sc_sym)
r2_full_sym = _np.array(r2_full_sym)
unique_anat_sym = _np.maximum(0, r2_full_sym - r2_sc_sym)
unique_sc_sym   = _np.maximum(0, r2_full_sym - r2_anat_sym)
shared_sym      = _np.maximum(0, r2_anat_sym + r2_sc_sym - r2_full_sym)

print(f"\n=== Bayesian variance decomposition (TARGET = FC, {K_PCA_FC_TGT} FC PCA components) ===")
print(f"  R-squared anatomy-only   mean={r2_anat_sym.mean():.4f}  sum={r2_anat_sym.sum():.4f}")
print(f"  R-squared SC-only        mean={r2_sc_sym.mean():.4f}    sum={r2_sc_sym.sum():.4f}")
print(f"  R-squared anatomy + SC   mean={r2_full_sym.mean():.4f}  sum={r2_full_sym.sum():.4f}")
print()
print(f"  Unique to anatomy        mean={unique_anat_sym.mean():.4f}  sum={unique_anat_sym.sum():.4f}")
print(f"  Unique to SC             mean={unique_sc_sym.mean():.4f}    sum={unique_sc_sym.sum():.4f}")
print(f"  Shared anatomy+SC        mean={shared_sym.mean():.4f}       sum={shared_sym.sum():.4f}")
print()
print(f"=== ASYMMETRY IN VARIANCE-PARTITION SPACE (4.3 vs 4.4) ===")
print(f"  Step 4.3 (target=SC): Unique-to-FC mean R-squared = 0.0205")
print(f"  Step 4.4 (target=FC): Unique-to-SC mean R-squared = {unique_sc_sym.mean():.4f}")
print(f"  Ratio                                              = {0.0205 / max(unique_sc_sym.mean(), 1e-12):.2f}x")
print()
print(f"Interpretation:")
print(f"  Ratio >> 1 -> FC->SC asymmetry confirmed in variance-partition space")
print(f"  Ratio ~ 1  -> the two directions are symmetric in unique cross-modal contribution")
print(f"  Ratio << 1 -> reverse asymmetry (unexpected)")
print()
print(f"  Top 5 FC components ranked by 'Unique to SC':")
for idx in _np.argsort(unique_sc_sym)[::-1][:5]:
    print(f"    PC{idx:3d}: anat={r2_anat_sym[idx]:.3f}  SC={r2_sc_sym[idx]:.3f}  full={r2_full_sym[idx]:.3f}  unique-SC={unique_sc_sym[idx]:.3f}")


Reducing FC -> 64 PCA components (target), SC -> 256 (predictor)...
feature counts: anatomy=16, SC_PCA=256, total=272

Fitting Bayesian Ridge for 64 FC components x 3 models each (192 fits)...

=== Bayesian variance decomposition (TARGET = FC, 64 FC PCA components) ===
  R-squared anatomy-only   mean=0.0087  sum=0.5594
  R-squared SC-only        mean=0.0265    sum=1.6963
  R-squared anatomy + SC   mean=0.0269  sum=1.7203

  Unique to anatomy        mean=0.0012  sum=0.0793
  Unique to SC             mean=0.0196    sum=1.2542
  Shared anatomy+SC        mean=0.0086       sum=0.5514

=== ASYMMETRY IN VARIANCE-PARTITION SPACE (4.3 vs 4.4) ===
  Step 4.3 (target=SC): Unique-to-FC mean R-squared = 0.0205
  Step 4.4 (target=FC): Unique-to-SC mean R-squared = 0.0196
  Ratio                                              = 1.05x

Interpretation:
  Ratio >> 1 -> FC->SC asymmetry confirmed in variance-partition space
  Ratio ~ 1  -> the two directions are symmetric in unique cross-modal contribution

In [20]:
# ============= STEP 5: RECONCILE 1.39x vs 1.05x DISCREPANCY =============
# We have:
#   3.5/3.6 (PLS + residual + edge demeaned-r)      -> ratio = 1.39x
#   4.3/4.4 (BayesianRidge + partition + latent R^2) -> ratio = 1.05x
#
# These differ in THREE dimensions: model class (PLS vs BR), method (residual vs
# partition), and aggregation (edge demeaned-r vs per-component R^2). To isolate
# which dimension causes the gap, run the two MISSING combinations:
#
#   Exp 3: PLS + partition + latent R^2          (swap BR -> PLS in 4.3/4.4)
#   Exp 4: BR  + residual  + edge demeaned-r     (swap PLS -> BR in 3.5/3.6)
#
# 4-combination logic:
#   If Exp3 ~ 1.05x and Exp4 ~ 1.39x:
#       -> MODEL CLASS doesn't matter; framework/aggregation drives the gap.
#   If Exp3 ~ 1.39x and Exp4 ~ 1.05x:
#       -> MODEL CLASS matters; PLS gives 1.39x, BR gives 1.05x.
#   Other patterns: multiple dimensions contribute.
import numpy as _np
import torch as _torch
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import BayesianRidge, LinearRegression
from models.eval.metrics import compute_demeaned_pearson_r

# Requires 3.5/3.6 vars in scope: _SC_train/test, _FC_train/test, _X_brainvol_train/test,
# _SC_resid_train/test, _FC_resid_train/test
assert '_SC_resid_train' in dir() and '_FC_resid_train' in dir(), \
    "Run STEP 3.5 and STEP 3.6 first (need residual arrays in scope)."

K_SRC = 256   # source PCA dim (matches 4.3/4.4 source)
K_TGT = 64    # target PCA dim (matches 4.3/4.4 target)
print(f"Using K_SRC={K_SRC} (predictor PCA), K_TGT={K_TGT} (target PCA)")
print("Building 4 PCAs (4 of FC/SC train; ~20s)...")
pca_fc_src = PCA(n_components=K_SRC, random_state=0).fit(_FC_train)
pca_sc_src = PCA(n_components=K_SRC, random_state=0).fit(_SC_train)
pca_sc_tgt = PCA(n_components=K_TGT, random_state=0).fit(_SC_train)
pca_fc_tgt = PCA(n_components=K_TGT, random_state=0).fit(_FC_train)

Z_FC_train_src = pca_fc_src.transform(_FC_train)
Z_FC_test_src  = pca_fc_src.transform(_FC_test)
Z_SC_train_src = pca_sc_src.transform(_SC_train)
Z_SC_test_src  = pca_sc_src.transform(_SC_test)
Z_SC_train_tgt = pca_sc_tgt.transform(_SC_train)
Z_SC_test_tgt  = pca_sc_tgt.transform(_SC_test)
Z_FC_train_tgt = pca_fc_tgt.transform(_FC_train)
Z_FC_test_tgt  = pca_fc_tgt.transform(_FC_test)

X_anat_tr = _X_brainvol_train
X_anat_te = _X_brainvol_test

def _per_comp_r2(Y_true, Y_pred):
    ss_total = ((Y_true - Y_true.mean(axis=0)) ** 2).sum(axis=0)
    ss_resid = ((Y_true - Y_pred) ** 2).sum(axis=0)
    return _np.maximum(0, 1 - ss_resid / _np.maximum(ss_total, 1e-12))

# ============= EXPERIMENT 3: PLS-based variance partition =============
print("\n=== EXPERIMENT 3: PLS partition (PLS instead of BR in 4.3/4.4 setup) ===")

def _pls_partition(X_anat_tr, X_xmod_tr, Y_tr, X_anat_te, X_xmod_te, Y_te):
    n_anat = X_anat_tr.shape[1]
    n_xmod = X_xmod_tr.shape[1]
    n_tgt  = Y_tr.shape[1]
    X_full_tr = _np.concatenate([X_anat_tr, X_xmod_tr], axis=1)
    X_full_te = _np.concatenate([X_anat_te, X_xmod_te], axis=1)
    k_a = min(n_anat, n_tgt)
    k_x = min(n_xmod, n_tgt)
    k_f = min(n_anat + n_xmod, n_tgt)
    pls_a = PLSRegression(n_components=k_a, scale=True).fit(X_anat_tr, Y_tr)
    pls_x = PLSRegression(n_components=k_x, scale=True).fit(X_xmod_tr, Y_tr)
    pls_f = PLSRegression(n_components=k_f, scale=True).fit(X_full_tr, Y_tr)
    pred_a = pls_a.predict(X_anat_te)
    pred_x = pls_x.predict(X_xmod_te)
    pred_f = pls_f.predict(X_full_te)
    r2_a = _per_comp_r2(Y_te, pred_a)
    r2_x = _per_comp_r2(Y_te, pred_x)
    r2_f = _per_comp_r2(Y_te, pred_f)
    unique_x = _np.maximum(0, r2_f - r2_a)
    return r2_a, r2_x, r2_f, unique_x

print("  Target = SC...")
r2_a_sc, r2_fc_sc, r2_full_sc_pls, unique_fc_sc_pls = _pls_partition(
    X_anat_tr, Z_FC_train_src, Z_SC_train_tgt,
    X_anat_te, Z_FC_test_src,  Z_SC_test_tgt,
)
print(f"    R^2 anat-only mean: {r2_a_sc.mean():.4f}  FC-only mean: {r2_fc_sc.mean():.4f}  full mean: {r2_full_sc_pls.mean():.4f}")
print(f"    Unique-to-FC mean R^2 = {unique_fc_sc_pls.mean():.4f}  (compare 4.3 BR: 0.0205)")

print("  Target = FC...")
r2_a_fc, r2_sc_fc, r2_full_fc_pls, unique_sc_fc_pls = _pls_partition(
    X_anat_tr, Z_SC_train_src, Z_FC_train_tgt,
    X_anat_te, Z_SC_test_src,  Z_FC_test_tgt,
)
print(f"    R^2 anat-only mean: {r2_a_fc.mean():.4f}  SC-only mean: {r2_sc_fc.mean():.4f}  full mean: {r2_full_fc_pls.mean():.4f}")
print(f"    Unique-to-SC mean R^2 = {unique_sc_fc_pls.mean():.4f}  (compare 4.4 BR: 0.0196)")

ratio_exp3 = unique_fc_sc_pls.mean() / max(unique_sc_fc_pls.mean(), 1e-12)
print(f"\n  -> EXP 3 RATIO (PLS partition): {ratio_exp3:.2f}x   [4.3/4.4 BR partition was 1.05x]")

# ============= EXPERIMENT 4: BR residual + edge-space demeaned-r =============
print("\n=== EXPERIMENT 4: BayesianRidge residual + edge-space demeaned-r ===")

# Residualize target PCA against brain-vol (in PCA space)
bvreg_sc_pca = LinearRegression().fit(X_anat_tr, Z_SC_train_tgt)
Z_SC_resid_train_pca = Z_SC_train_tgt - bvreg_sc_pca.predict(X_anat_tr)
Z_SC_resid_test_pca  = Z_SC_test_tgt  - bvreg_sc_pca.predict(X_anat_te)

bvreg_fc_pca = LinearRegression().fit(X_anat_tr, Z_FC_train_tgt)
Z_FC_resid_train_pca = Z_FC_train_tgt - bvreg_fc_pca.predict(X_anat_tr)
Z_FC_resid_test_pca  = Z_FC_test_tgt  - bvreg_fc_pca.predict(X_anat_te)

def _br_residual_edge(X_xmod_tr, Y_resid_pca_tr, X_xmod_te, pca_tgt, Y_resid_edge_te, Y_resid_train_mean_edge):
    """BayesianRidge per target PCA component on residual; decode to edges; demeaned-r."""
    n_tgt = Y_resid_pca_tr.shape[1]
    pred_pca = _np.zeros((X_xmod_te.shape[0], n_tgt), dtype=_np.float32)
    for k in range(n_tgt):
        m = BayesianRidge(max_iter=300).fit(X_xmod_tr, Y_resid_pca_tr[:, k])
        pred_pca[:, k] = m.predict(X_xmod_te)
    pred_edges = pca_tgt.inverse_transform(pred_pca).astype(_np.float32)
    return float(compute_demeaned_pearson_r(
        _torch.tensor(pred_edges, dtype=_torch.float32),
        _torch.tensor(Y_resid_edge_te.astype(_np.float32), dtype=_torch.float32),
        _torch.tensor(Y_resid_train_mean_edge.astype(_np.float32), dtype=_torch.float32),
    ))

print("  FC -> SC_residual (BR per SC PCA component)...")
d_br_sc = _br_residual_edge(
    Z_FC_train_src, Z_SC_resid_train_pca, Z_FC_test_src,
    pca_sc_tgt, _SC_resid_test, _SC_resid_train.mean(axis=0),
)
print(f"    demeaned-r = {d_br_sc:.4f}  (compare 3.5 PLS-resid: 0.0778)")

print("  SC -> FC_residual (BR per FC PCA component)...")
d_br_fc = _br_residual_edge(
    Z_SC_train_src, Z_FC_resid_train_pca, Z_SC_test_src,
    pca_fc_tgt, _FC_resid_test, _FC_resid_train.mean(axis=0),
)
print(f"    demeaned-r = {d_br_fc:.4f}  (compare 3.6 PLS-resid: 0.0561)")

ratio_exp4 = d_br_sc / max(d_br_fc, 1e-12)
print(f"\n  -> EXP 4 RATIO (BR residual, edge demeaned-r): {ratio_exp4:.2f}x   [3.5/3.6 PLS residual was 1.39x]")

# ============= SUMMARY =============
print("\n\n" + "=" * 70)
print("                SUMMARY: 4 COMBINATIONS (target=SC : target=FC)         ")
print("=" * 70)
print(f"  3.5/3.6  (PLS  + residual  + edge demeaned-r) : 1.39x  [existing]")
print(f"  4.3/4.4  (BR   + partition + latent R^2)      : 1.05x  [existing]")
print(f"  Exp 3    (PLS  + partition + latent R^2)      : {ratio_exp3:.2f}x  [NEW]")
print(f"  Exp 4    (BR   + residual  + edge demeaned-r) : {ratio_exp4:.2f}x  [NEW]")
print()
print("INTERPRETATION:")
if abs(ratio_exp3 - 1.05) < 0.2 and abs(ratio_exp4 - 1.39) < 0.2:
    print("  -> METHOD/AGGREGATION drives the gap.")
    print("     PLS/BR don't differ; what matters is residual-edge-demeaned-r vs partition-latent-R^2.")
elif abs(ratio_exp3 - 1.39) < 0.2 and abs(ratio_exp4 - 1.05) < 0.2:
    print("  -> MODEL CLASS drives it.")
    print("     PLS gives 1.39x regardless of framework; BR gives 1.05x regardless.")
else:
    print("  -> Multiple dimensions contribute. Need finer decomposition.")
    print(f"     Exp3 is {ratio_exp3:.2f}x (PLS partition); Exp4 is {ratio_exp4:.2f}x (BR residual).")


Using K_SRC=256 (predictor PCA), K_TGT=64 (target PCA)
Building 4 PCAs (4 of FC/SC train; ~20s)...

=== EXPERIMENT 3: PLS partition (PLS instead of BR in 4.3/4.4 setup) ===
  Target = SC...


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


    R^2 anat-only mean: 0.0270  FC-only mean: 0.0149  full mean: 0.0220
    Unique-to-FC mean R^2 = 0.0053  (compare 4.3 BR: 0.0205)
  Target = FC...


/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)
/ext3/miniforge3/envs/kraken_env/lib/python3.12/site-packages/sklearn/cross_decomposition/_pls.py:104: ConvergenceWarning: Maximum number of iterations reached
  warnings.warn("Maximum number of iterations reached", ConvergenceWarning)


    R^2 anat-only mean: 0.0043  SC-only mean: 0.0006  full mean: 0.0003
    Unique-to-SC mean R^2 = 0.0003  (compare 4.4 BR: 0.0196)

  -> EXP 3 RATIO (PLS partition): 18.24x   [4.3/4.4 BR partition was 1.05x]

=== EXPERIMENT 4: BayesianRidge residual + edge-space demeaned-r ===
  FC -> SC_residual (BR per SC PCA component)...
    demeaned-r = 0.0111  (compare 3.5 PLS-resid: 0.0778)
  SC -> FC_residual (BR per FC PCA component)...
    demeaned-r = 0.0066  (compare 3.6 PLS-resid: 0.0561)

  -> EXP 4 RATIO (BR residual, edge demeaned-r): 1.68x   [3.5/3.6 PLS residual was 1.39x]


                SUMMARY: 4 COMBINATIONS (target=SC : target=FC)         
  3.5/3.6  (PLS  + residual  + edge demeaned-r) : 1.39x  [existing]
  4.3/4.4  (BR   + partition + latent R^2)      : 1.05x  [existing]
  Exp 3    (PLS  + partition + latent R^2)      : 18.24x  [NEW]
  Exp 4    (BR   + residual  + edge demeaned-r) : 1.68x  [NEW]

INTERPRETATION:
  -> Multiple dimensions contribute. Need finer decomposition.

In [28]:
# ============= STEP 5.1: CONTROLLED 2x2 AT K_TGT=256 (FULL-PANEL RETROFIT) =============
# Fixes Step 5's flaws AND adds the full 6-metric panel (2026-05-26 retrofit):
#   - PLS uses max_iter=2000 and n_components=64 (not over-parametrized)
#   - K_TGT held FIXED at 256 across all 4 combinations (matches 3.5/3.6)
#   - K_SRC held FIXED at 256
#   - Residual and partition each use their natural PCA basis (residual: PCA on
#     residuals; partition: PCA on raw target). This is a property of the
#     framework, not a confound.
#   - A/C (edge-space): report all 6 metrics. Ratio quoted on demeaned_pearson
#     (the headline) but top1_acc and avg_rank give independent corroboration.
#   - B/D (latent-space): R^2 stays primary; also report latent-space mean
#     pearson per component as a non-R^2 sanity check.
#
# 4 combinations to isolate model class vs framework:
#   A: PLS  + residual  + edge demeaned-r   (= 3.5/3.6 setup; sanity vs 1.39x)
#   B: PLS  + partition + latent R^2        (PLS in partition framework)
#   C: BR   + residual  + edge demeaned-r   (BR in residual framework)
#   D: BR   + partition + latent R^2        (= 4.3/4.4 but at K_TGT=256)
import numpy as _np
import torch as _torch
from sklearn.decomposition import PCA
from sklearn.cross_decomposition import PLSRegression
from sklearn.linear_model import BayesianRidge, LinearRegression
from models.eval.metrics import compute_demeaned_pearson_r

assert '_SC_resid_train' in dir() and '_FC_resid_train' in dir(), \
    "Run STEP 3.5 and STEP 3.6 first (need _SC_resid_train/test and _FC_resid_train/test)."
assert '_full_panel_eval' in dir(), \
    "Run the FULL-PANEL HELPER cell (between Step 2e and Step 3) first."

K_SRC = 256
K_TGT = 256
PLS_NCOMP = 64
PLS_MAX_ITER = 2000

print(f"Setup: K_SRC={K_SRC}, K_TGT={K_TGT}, PLS_NCOMP={PLS_NCOMP}, PLS_MAX_ITER={PLS_MAX_ITER}")

# PCAs for partition (B, D): PCA on RAW target
print("\nBuilding PCAs on raw FC/SC (for partition B/D)...")
pca_fc_raw = PCA(n_components=K_TGT, random_state=0).fit(_FC_train)
pca_sc_raw = PCA(n_components=K_TGT, random_state=0).fit(_SC_train)
Z_FC_tr_raw = pca_fc_raw.transform(_FC_train)
Z_FC_te_raw = pca_fc_raw.transform(_FC_test)
Z_SC_tr_raw = pca_sc_raw.transform(_SC_train)
Z_SC_te_raw = pca_sc_raw.transform(_SC_test)

# PCAs for residual (A, C): PCA on RESIDUALS (matches 3.5/3.6 exactly)
print("Building PCAs on edge-space residuals (for residual A/C, matches 3.5/3.6)...")
pca_sc_resid = PCA(n_components=K_TGT, random_state=0).fit(_SC_resid_train)
pca_fc_resid = PCA(n_components=K_TGT, random_state=0).fit(_FC_resid_train)
# Source-side PCAs (same as raw -- source isn't residualized)
pca_fc_src = pca_fc_raw
pca_sc_src = pca_sc_raw
Z_FC_tr_src = Z_FC_tr_raw
Z_FC_te_src = Z_FC_te_raw
Z_SC_tr_src = Z_SC_tr_raw
Z_SC_te_src = Z_SC_te_raw
Z_SC_resid_tr = pca_sc_resid.transform(_SC_resid_train)
Z_SC_resid_te = pca_sc_resid.transform(_SC_resid_test)
Z_FC_resid_tr = pca_fc_resid.transform(_FC_resid_train)
Z_FC_resid_te = pca_fc_resid.transform(_FC_resid_test)

X_anat_tr = _X_brainvol_train
X_anat_te = _X_brainvol_test


def _per_comp_r2(Y_true, Y_pred):
    ss_total = ((Y_true - Y_true.mean(axis=0)) ** 2).sum(axis=0)
    ss_resid = ((Y_true - Y_pred) ** 2).sum(axis=0)
    return _np.maximum(0, 1 - ss_resid / _np.maximum(ss_total, 1e-12))


def _per_comp_pearson(Y_true, Y_pred):
    """Per-component (column-wise) Pearson r between latent target and prediction."""
    yt = Y_true - Y_true.mean(axis=0)
    yp = Y_pred - Y_pred.mean(axis=0)
    num = (yt * yp).sum(axis=0)
    den = _np.sqrt((yt ** 2).sum(axis=0) * (yp ** 2).sum(axis=0))
    return num / _np.maximum(den, 1e-12)


# ---------- A. PLS RESIDUAL + EDGE (full 6-metric panel) ----------
print("\n" + "=" * 72)
print("A. PLS RESIDUAL + EDGE   (full panel; sanity-check vs 3.5/3.6 = 1.39x demeaned)")
print("=" * 72)

def _pls_resid_edge_full(X_src_tr, Y_resid_pca_tr, X_src_te,
                          pca_resid, Y_resid_edge_te, train_mean_edge):
    """Returns (panel_dict, predicted_edges) so we can full-panel evaluate."""
    pls = PLSRegression(n_components=PLS_NCOMP, scale=True, max_iter=PLS_MAX_ITER) \
        .fit(X_src_tr, Y_resid_pca_tr)
    pred_pca = pls.predict(X_src_te)
    pred_edges = pca_resid.inverse_transform(pred_pca).astype(_np.float32)
    panel = _full_panel_eval(pred_edges, Y_resid_edge_te, train_mean_edge)
    return panel, pred_edges

print("  Fitting FC -> SC_resid (PLS)...")
panel_A_FC_SC, _pred_A_FC_SC = _pls_resid_edge_full(
    Z_FC_tr_src, Z_SC_resid_tr, Z_FC_te_src,
    pca_sc_resid, _SC_resid_test, _SC_resid_train.mean(axis=0))
print(_fmt_panel(panel_A_FC_SC, "FC -> SC_resid              "))

print("  Fitting SC -> FC_resid (PLS)...")
panel_A_SC_FC, _pred_A_SC_FC = _pls_resid_edge_full(
    Z_SC_tr_src, Z_FC_resid_tr, Z_SC_te_src,
    pca_fc_resid, _FC_resid_test, _FC_resid_train.mean(axis=0))
print(_fmt_panel(panel_A_SC_FC, "SC -> FC_resid              "))

# Headline ratio: demeaned_pearson (matches 3.5/3.6 = 1.39x sanity reference)
d_A_FC_SC = panel_A_FC_SC['demeaned_pearson']
d_A_SC_FC = panel_A_SC_FC['demeaned_pearson']
ratio_A = d_A_FC_SC / max(d_A_SC_FC, 1e-12)
print(f"  -> RATIO A (demeaned_pearson) = {ratio_A:.3f}x   (expected ~1.39x if matches 3.5/3.6)")
print(f"     ratio top1_acc             = {panel_A_FC_SC['top1_acc']/max(panel_A_SC_FC['top1_acc'],1e-12):.3f}x")
print(f"     ratio avg_rank             = {panel_A_FC_SC['avg_rank']/max(panel_A_SC_FC['avg_rank'],1e-12):.3f}x")


# ---------- C. BR RESIDUAL + EDGE (full 6-metric panel) ----------
print("\n" + "=" * 72)
print("C. BR RESIDUAL + EDGE   (full panel; isolates model class vs A)")
print("=" * 72)

def _br_resid_edge_full(X_src_tr, Y_resid_pca_tr, X_src_te,
                         pca_resid, Y_resid_edge_te, train_mean_edge):
    n_tgt = Y_resid_pca_tr.shape[1]
    pred_pca = _np.zeros((X_src_te.shape[0], n_tgt), dtype=_np.float32)
    for k in range(n_tgt):
        m = BayesianRidge(max_iter=300).fit(X_src_tr, Y_resid_pca_tr[:, k])
        pred_pca[:, k] = m.predict(X_src_te)
    pred_edges = pca_resid.inverse_transform(pred_pca).astype(_np.float32)
    panel = _full_panel_eval(pred_edges, Y_resid_edge_te, train_mean_edge)
    return panel, pred_edges

print(f"  Fitting FC -> SC_resid ({K_TGT} BR fits)...")
panel_C_FC_SC, _pred_C_FC_SC = _br_resid_edge_full(
    Z_FC_tr_src, Z_SC_resid_tr, Z_FC_te_src,
    pca_sc_resid, _SC_resid_test, _SC_resid_train.mean(axis=0))
print(_fmt_panel(panel_C_FC_SC, "FC -> SC_resid              "))

print(f"  Fitting SC -> FC_resid ({K_TGT} BR fits)...")
panel_C_SC_FC, _pred_C_SC_FC = _br_resid_edge_full(
    Z_SC_tr_src, Z_FC_resid_tr, Z_SC_te_src,
    pca_fc_resid, _FC_resid_test, _FC_resid_train.mean(axis=0))
print(_fmt_panel(panel_C_SC_FC, "SC -> FC_resid              "))

d_C_FC_SC = panel_C_FC_SC['demeaned_pearson']
d_C_SC_FC = panel_C_SC_FC['demeaned_pearson']
ratio_C = d_C_FC_SC / max(d_C_SC_FC, 1e-12)
print(f"  -> RATIO C (demeaned_pearson) = {ratio_C:.3f}x")
print(f"     ratio top1_acc             = {panel_C_FC_SC['top1_acc']/max(panel_C_SC_FC['top1_acc'],1e-12):.3f}x")
print(f"     ratio avg_rank             = {panel_C_FC_SC['avg_rank']/max(panel_C_SC_FC['avg_rank'],1e-12):.3f}x")


# ---------- B. PLS PARTITION + LATENT (R^2 primary, +latent pearson) ----------
print("\n" + "=" * 72)
print("B. PLS PARTITION + LATENT   (isolates framework vs A; +latent pearson sanity)")
print("=" * 72)

def _pls_partition_latent_full(X_anat_tr, X_xmod_tr, Y_tr, X_anat_te, X_xmod_te, Y_te):
    anat_n = min(X_anat_tr.shape[1], Y_tr.shape[1])
    X_full_tr = _np.concatenate([X_anat_tr, X_xmod_tr], axis=1)
    X_full_te = _np.concatenate([X_anat_te, X_xmod_te], axis=1)
    pls_a = PLSRegression(n_components=anat_n,    scale=True, max_iter=PLS_MAX_ITER).fit(X_anat_tr, Y_tr)
    pls_x = PLSRegression(n_components=PLS_NCOMP, scale=True, max_iter=PLS_MAX_ITER).fit(X_xmod_tr, Y_tr)
    pls_f = PLSRegression(n_components=PLS_NCOMP, scale=True, max_iter=PLS_MAX_ITER).fit(X_full_tr, Y_tr)
    pred_a = pls_a.predict(X_anat_te); pred_x = pls_x.predict(X_xmod_te); pred_f = pls_f.predict(X_full_te)
    r2_a = _per_comp_r2(Y_te, pred_a); r2_x = _per_comp_r2(Y_te, pred_x); r2_f = _per_comp_r2(Y_te, pred_f)
    pr_a = _per_comp_pearson(Y_te, pred_a); pr_x = _per_comp_pearson(Y_te, pred_x); pr_f = _per_comp_pearson(Y_te, pred_f)
    return dict(r2_a=r2_a.mean(), r2_x=r2_x.mean(), r2_f=r2_f.mean(),
                r2_uniq=_np.maximum(0, r2_f - r2_a).mean(),
                pr_a=pr_a.mean(), pr_x=pr_x.mean(), pr_f=pr_f.mean())

print(f"  Target = SC ({K_TGT} components)...")
B_sc = _pls_partition_latent_full(X_anat_tr, Z_FC_tr_raw, Z_SC_tr_raw, X_anat_te, Z_FC_te_raw, Z_SC_te_raw)
print(f"    R^2:     anat={B_sc['r2_a']:.4f}  FC-only={B_sc['r2_x']:.4f}  full={B_sc['r2_f']:.4f}  unique-FC={B_sc['r2_uniq']:.4f}")
print(f"    Pearson: anat={B_sc['pr_a']:+.4f}  FC-only={B_sc['pr_x']:+.4f}  full={B_sc['pr_f']:+.4f}  (latent-space, per-component mean)")
print(f"  Target = FC ({K_TGT} components)...")
B_fc = _pls_partition_latent_full(X_anat_tr, Z_SC_tr_raw, Z_FC_tr_raw, X_anat_te, Z_SC_te_raw, Z_FC_te_raw)
print(f"    R^2:     anat={B_fc['r2_a']:.4f}  SC-only={B_fc['r2_x']:.4f}  full={B_fc['r2_f']:.4f}  unique-SC={B_fc['r2_uniq']:.4f}")
print(f"    Pearson: anat={B_fc['pr_a']:+.4f}  SC-only={B_fc['pr_x']:+.4f}  full={B_fc['pr_f']:+.4f}")
ratio_B    = B_sc['r2_uniq']    / max(B_fc['r2_uniq'],    1e-12)
ratio_B_pr = B_sc['pr_x']       / max(B_fc['pr_x'],       1e-12)
print(f"  -> RATIO B (R^2_unique)     = {ratio_B:.3f}x")
print(f"     ratio latent pearson(xmod-only) = {ratio_B_pr:.3f}x")


# ---------- D. BR PARTITION + LATENT (R^2 primary, +latent pearson) ----------
print("\n" + "=" * 72)
print(f"D. BR PARTITION + LATENT at K_TGT={K_TGT}   (vs 4.3/4.4 at K_TGT=64 = 1.05x)")
print("=" * 72)

def _br_partition_latent_full(X_anat_tr, X_xmod_tr, Y_tr, X_anat_te, X_xmod_te, Y_te):
    X_full_tr = _np.concatenate([X_anat_tr, X_xmod_tr], axis=1)
    X_full_te = _np.concatenate([X_anat_te, X_xmod_te], axis=1)
    n_tgt = Y_tr.shape[1]
    r2_a = _np.zeros(n_tgt); r2_x = _np.zeros(n_tgt); r2_f = _np.zeros(n_tgt)
    pr_a = _np.zeros(n_tgt); pr_x = _np.zeros(n_tgt); pr_f = _np.zeros(n_tgt)
    for k in range(n_tgt):
        y_tr = Y_tr[:, k]; y_te = Y_te[:, k]
        ss = ((y_te - y_te.mean()) ** 2).sum()
        if ss < 1e-12: continue
        pa = BayesianRidge(max_iter=300).fit(X_anat_tr, y_tr).predict(X_anat_te)
        px = BayesianRidge(max_iter=300).fit(X_xmod_tr, y_tr).predict(X_xmod_te)
        pf = BayesianRidge(max_iter=300).fit(X_full_tr, y_tr).predict(X_full_te)
        r2_a[k] = max(0, 1 - ((y_te - pa) ** 2).sum() / ss)
        r2_x[k] = max(0, 1 - ((y_te - px) ** 2).sum() / ss)
        r2_f[k] = max(0, 1 - ((y_te - pf) ** 2).sum() / ss)
        yc = y_te - y_te.mean()
        pr_a[k] = (yc * (pa - pa.mean())).sum() / max(_np.sqrt((yc**2).sum() * ((pa-pa.mean())**2).sum()), 1e-12)
        pr_x[k] = (yc * (px - px.mean())).sum() / max(_np.sqrt((yc**2).sum() * ((px-px.mean())**2).sum()), 1e-12)
        pr_f[k] = (yc * (pf - pf.mean())).sum() / max(_np.sqrt((yc**2).sum() * ((pf-pf.mean())**2).sum()), 1e-12)
    return dict(r2_a=r2_a.mean(), r2_x=r2_x.mean(), r2_f=r2_f.mean(),
                r2_uniq=_np.maximum(0, r2_f - r2_a).mean(),
                pr_a=pr_a.mean(), pr_x=pr_x.mean(), pr_f=pr_f.mean())

print(f"  Target = SC ({K_TGT} components x 3 BR fits = {3*K_TGT} fits)...")
D_sc = _br_partition_latent_full(X_anat_tr, Z_FC_tr_raw, Z_SC_tr_raw, X_anat_te, Z_FC_te_raw, Z_SC_te_raw)
print(f"    R^2:     anat={D_sc['r2_a']:.4f}  FC-only={D_sc['r2_x']:.4f}  full={D_sc['r2_f']:.4f}  unique-FC={D_sc['r2_uniq']:.4f}")
print(f"    Pearson: anat={D_sc['pr_a']:+.4f}  FC-only={D_sc['pr_x']:+.4f}  full={D_sc['pr_f']:+.4f}")
print(f"  Target = FC ({3*K_TGT} BR fits)...")
D_fc = _br_partition_latent_full(X_anat_tr, Z_SC_tr_raw, Z_FC_tr_raw, X_anat_te, Z_SC_te_raw, Z_FC_te_raw)
print(f"    R^2:     anat={D_fc['r2_a']:.4f}  SC-only={D_fc['r2_x']:.4f}  full={D_fc['r2_f']:.4f}  unique-SC={D_fc['r2_uniq']:.4f}")
print(f"    Pearson: anat={D_fc['pr_a']:+.4f}  SC-only={D_fc['pr_x']:+.4f}  full={D_fc['pr_f']:+.4f}")
ratio_D    = D_sc['r2_uniq'] / max(D_fc['r2_uniq'], 1e-12)
ratio_D_pr = D_sc['pr_x']    / max(D_fc['pr_x'],    1e-12)
print(f"  -> RATIO D (R^2_unique)     = {ratio_D:.3f}x")
print(f"     ratio latent pearson(xmod-only) = {ratio_D_pr:.3f}x")


# ---------- SUMMARY (full panel) ----------
print("\n\n" + "=" * 78)
print(f"  SUMMARY: CONTROLLED 2x2 AT K_TGT={K_TGT}, K_SRC={K_SRC}, PLS_NCOMP={PLS_NCOMP}")
print("=" * 78)
print("  -- A/C (residual + edge): full 6-metric panel per direction --")
print()
print(f"  A (PLS, FC->SC_resid): {_fmt_panel(panel_A_FC_SC, '')}")
print(f"  A (PLS, SC->FC_resid): {_fmt_panel(panel_A_SC_FC, '')}")
print(f"  C (BR,  FC->SC_resid): {_fmt_panel(panel_C_FC_SC, '')}")
print(f"  C (BR,  SC->FC_resid): {_fmt_panel(panel_C_SC_FC, '')}")
print()
print("  -- Asymmetry ratios (FC->SC / SC->FC) on each metric --")
print()
print("                          demeaned_r   top1_acc   avg_rank   pearson    r2         mse")
for name, fc_sc, sc_fc in [
    ("  A (PLS, residual)", panel_A_FC_SC, panel_A_SC_FC),
    ("  C (BR,  residual)", panel_C_FC_SC, panel_C_SC_FC),
]:
    print(f"{name}: "
          f"{fc_sc['demeaned_pearson']/max(sc_fc['demeaned_pearson'],1e-12):>6.3f}x     "
          f"{fc_sc['top1_acc']/max(sc_fc['top1_acc'],1e-12):>6.3f}x    "
          f"{fc_sc['avg_rank']/max(sc_fc['avg_rank'],1e-12):>6.3f}x    "
          f"{fc_sc['pearson']/max(sc_fc['pearson'],1e-12):>6.3f}x    "
          f"{fc_sc['r2']/max(abs(sc_fc['r2']),1e-12):>+6.3f}    "
          f"{sc_fc['mse']/max(fc_sc['mse'],1e-12):>6.3f}x  (mse: lower-is-better, ratio shown SC->FC/FC->SC)")
print()
print("  -- B/D (partition + latent): primary R^2, +latent pearson cross-check --")
print()
print(f"  B (PLS, SC target): R^2_unique={B_sc['r2_uniq']:.4f}   latent_pr_xmod={B_sc['pr_x']:+.4f}")
print(f"  B (PLS, FC target): R^2_unique={B_fc['r2_uniq']:.4f}   latent_pr_xmod={B_fc['pr_x']:+.4f}")
print(f"  D (BR,  SC target): R^2_unique={D_sc['r2_uniq']:.4f}   latent_pr_xmod={D_sc['pr_x']:+.4f}")
print(f"  D (BR,  FC target): R^2_unique={D_fc['r2_uniq']:.4f}   latent_pr_xmod={D_fc['pr_x']:+.4f}")
print()
print("                            R^2 ratio   latent pearson ratio (xmod-only)")
print(f"  B (PLS, partition):     {ratio_B:>6.3f}x      {ratio_B_pr:>+6.3f}")
print(f"  D (BR,  partition):     {ratio_D:>6.3f}x      {ratio_D_pr:>+6.3f}")
print()
print("REFERENCES:")
print(f"  3.5/3.6 (= A spec): 1.39x  -- A should reproduce this on demeaned_pearson")
print(f"  4.3/4.4 at K_TGT=64 (= D spec but lower K_TGT): 1.05x")
print()
print("DIAGNOSTIC LOGIC (demeaned_pearson ratio):")
def _near(a, b, tol=0.15): return abs(a - b) / max(abs(b), 1e-12) < tol
if _near(ratio_A, ratio_C) and _near(ratio_B, ratio_D):
    print("  -> MODEL CLASS doesn't matter (A~C, B~D). FRAMEWORK drives the ratio gap.")
elif _near(ratio_A, ratio_B) and _near(ratio_C, ratio_D):
    print("  -> FRAMEWORK doesn't matter (A~B, C~D). MODEL CLASS drives it.")
elif _near(ratio_A, ratio_D) and _near(ratio_B, ratio_C):
    print("  -> Both dimensions affect ratio but offset each other. Deeper investigation needed.")
else:
    print("  -> Multiple dimensions contribute non-trivially. Need finer decomposition.")
print()
print("CROSS-METRIC CONSISTENCY (the new question):")
print("  Compare demeaned_r ratio vs top1_acc / avg_rank ratios for A and C:")
print("    A demeaned={:.2f}x  top1={:.2f}x  avg_rank={:.2f}x".format(
    ratio_A,
    panel_A_FC_SC['top1_acc']/max(panel_A_SC_FC['top1_acc'],1e-12),
    panel_A_FC_SC['avg_rank']/max(panel_A_SC_FC['avg_rank'],1e-12)))
print("    C demeaned={:.2f}x  top1={:.2f}x  avg_rank={:.2f}x".format(
    ratio_C,
    panel_C_FC_SC['top1_acc']/max(panel_C_SC_FC['top1_acc'],1e-12),
    panel_C_FC_SC['avg_rank']/max(panel_C_SC_FC['avg_rank'],1e-12)))
print("  If all 3 ratios point in the same direction with similar magnitude,")
print("  the FC->SC > SC->FC story is metric-robust, not a demeaned-r artifact.")


Setup: K_SRC=256, K_TGT=256, PLS_NCOMP=64, PLS_MAX_ITER=2000

Building PCAs on raw FC/SC (for partition B/D)...
Building PCAs on edge-space residuals (for residual A/C, matches 3.5/3.6)...

A. PLS RESIDUAL + EDGE   (full panel; sanity-check vs 3.5/3.6 = 1.39x demeaned)
  Fitting FC -> SC_resid (PLS)...
  FC -> SC_resid                            mse=0.00551  r2=-0.0494  pearson=0.0777  demeaned=0.0777  top1=0.0821  avg_rank=0.8125
  Fitting SC -> FC_resid (PLS)...
  SC -> FC_resid                            mse=0.01428  r2=-0.0705  pearson=0.0560  demeaned=0.0560  top1=0.0154  avg_rank=0.6487
  -> RATIO A (demeaned_pearson) = 1.388x   (expected ~1.39x if matches 3.5/3.6)
     ratio top1_acc             = 5.333x
     ratio avg_rank             = 1.253x

C. BR RESIDUAL + EDGE   (full panel; isolates model class vs A)
  Fitting FC -> SC_resid (256 BR fits)...
  FC -> SC_resid                            mse=0.00529  r2=-0.0060  pearson=0.0931  demeaned=0.0931  top1=0.0564  avg_rank=0.8054


In [23]:
for name, run_out in {
    "CrossModalPCA": pca_run,
    "CrossModal_PLS_SVD": pls_svd_run,
    "CrossModal_PCA_PLS": pca_pls_run,
}.items():
    report_base = RESULTS_ROOT / f"{name}_eval_test"
    run_out["evaluators"]["test"].analyze_results(
        verbose=False,
        filepath=str(report_base),
        output_format="md",
        model_name=name,
    )
    print(f"wrote {report_base}.md")


Number of unique demographic categories: 14


/scratch/ans9868/Conn2Conn/models/eval/evaluator_viz_markdown.py:1190: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Markdown report saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModalPCA_eval_test.md
Figures saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModalPCA_eval_test_plots/
wrote results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModalPCA_eval_test.md
Number of unique demographic categories: 14


/scratch/ans9868/Conn2Conn/models/eval/evaluator_viz_markdown.py:1190: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Markdown report saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PLS_SVD_eval_test.md
Figures saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PLS_SVD_eval_test_plots/
wrote results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PLS_SVD_eval_test.md
Number of unique demographic categories: 14


/scratch/ans9868/Conn2Conn/models/eval/evaluator_viz_markdown.py:1190: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout()


Markdown report saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PCA_PLS_eval_test.md
Figures saved to: results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PCA_PLS_eval_test_plots/
wrote results/local_results/crossmodal_pca_pls_closed_form_overview/CrossModal_PCA_PLS_eval_test.md


In [29]:
# ============= STEP 5.2: CROSS-STEP FULL-PANEL SUMMARY =============
# Single consolidated view of every prediction we've made, on the same
# 6 metrics. Pulls from variables already populated by:
#   Step 3       : brain-vol -> SC          (_panel_step3)
#   Step 3.5     : brain-vol -> SC, FC -> SC raw, FC -> SC_resid
#   Step 3.6     : brain-vol -> FC, SC -> FC raw, SC -> FC_resid
#   Step 5.1     : A (PLS resid) + C (BR resid) for both directions
#
# Two grouping questions to read this table for:
#   1. Does the FC->SC > SC->FC asymmetry hold across all 6 metrics, or
#      only on demeaned_pearson?
#   2. Does brain-vol dominate FC on identifiability (top1/avg_rank) too,
#      or only on demeaned_pearson + variance metrics?
import pandas as _pd
assert '_panel_step3'   in dir(), "Run STEP 3 (and the full-panel block at its end) first."
assert '_panel_35_res'  in dir(), "Run STEP 3.5 (and the full-panel block at its end) first."
assert '_panel_36_res'  in dir(), "Run STEP 3.6 (and the full-panel block at its end) first."
assert 'panel_A_FC_SC'  in dir(), "Run STEP 5.1 first."

_rows = [
    ("brain-vol -> SC",                 "anatomy",   _panel_step3),
    ("brain-vol -> FC",                 "anatomy",   _panel_36_bv),
    ("FC -> SC raw (manual sanity)",    "raw FC->SC", _panel_35_raw),
    ("SC -> FC raw (manual sanity)",    "raw SC->FC", _panel_36_raw),
    ("FC -> SC_residual (PLS, 3.5)",    "resid FC->SC PLS", _panel_35_res),
    ("SC -> FC_residual (PLS, 3.6)",    "resid SC->FC PLS", _panel_36_res),
    ("FC -> SC_residual (PLS, 5.1 A)",  "resid FC->SC PLS (5.1)", panel_A_FC_SC),
    ("SC -> FC_residual (PLS, 5.1 A)",  "resid SC->FC PLS (5.1)", panel_A_SC_FC),
    ("FC -> SC_residual (BR,  5.1 C)",  "resid FC->SC BR",  panel_C_FC_SC),
    ("SC -> FC_residual (BR,  5.1 C)",  "resid SC->FC BR",  panel_C_SC_FC),
]
_df = _pd.DataFrame(
    [{**{"experiment": name, "group": grp}, **p} for name, grp, p in _rows]
)[["experiment", "group", "mse", "r2", "pearson", "demeaned_pearson", "top1_acc", "avg_rank"]]
print("\n=== CROSS-STEP FULL-PANEL TABLE ===\n")
print(_df.to_string(index=False, float_format=lambda x: f"{x:.4f}"))

# ---- Ratio table: every asymmetry (FC->SC / SC->FC) on every metric ----
def _ratio_row(label, fc_sc, sc_fc):
    return {
        "comparison": label,
        "demeaned_pearson": fc_sc["demeaned_pearson"] / max(sc_fc["demeaned_pearson"], 1e-12),
        "top1_acc":         fc_sc["top1_acc"]         / max(sc_fc["top1_acc"],         1e-12),
        "avg_rank":         fc_sc["avg_rank"]         / max(sc_fc["avg_rank"],         1e-12),
        "pearson":          fc_sc["pearson"]          / max(sc_fc["pearson"],          1e-12),
    }

_ratios = _pd.DataFrame([
    _ratio_row("raw FC->SC / SC->FC",                _panel_35_raw, _panel_36_raw),
    _ratio_row("resid PLS (3.5/3.6) FC->SC/SC->FC",  _panel_35_res, _panel_36_res),
    _ratio_row("resid PLS (5.1 A) FC->SC/SC->FC",    panel_A_FC_SC, panel_A_SC_FC),
    _ratio_row("resid BR  (5.1 C) FC->SC/SC->FC",    panel_C_FC_SC, panel_C_SC_FC),
])
print("\n=== ASYMMETRY RATIOS (FC->SC / SC->FC) ACROSS METRICS ===\n")
print(_ratios.to_string(index=False, float_format=lambda x: f"{x:.3f}x"))

# ---- Anatomy-vs-FC head-to-head on identifiability triad ----
print("\n=== ANATOMY vs FC ON IDENTIFIABILITY (the new question Step 3 retrofit asks) ===\n")
_idtri = _pd.DataFrame([
    {"predictor": "brain-vol  -> SC", **{k: _panel_step3[k]   for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "FC raw     -> SC", **{k: _panel_35_raw[k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "FC residual-> SC", **{k: _panel_35_res[k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "brain-vol  -> FC", **{k: _panel_36_bv[k]   for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "SC raw     -> FC", **{k: _panel_36_raw[k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
    {"predictor": "SC residual-> FC", **{k: _panel_36_res[k]  for k in ("demeaned_pearson","top1_acc","avg_rank")}},
])
print(_idtri.to_string(index=False, float_format=lambda x: f"{x:.4f}"))
print()
print("Reading hints:")
print("  - If brain-vol -> SC top1 << FC raw -> SC top1: anatomy story is demeaned-r-specific;")
print("    FC carries identifiability info anatomy can't reach.")
print("  - If FC residual -> SC top1 > 0.05 (~ chance for n=195): residual FC retains real ID power.")
print("  - If anatomy ratios across metrics are inconsistent: the asymmetry interpretation")
print("    depends on which metric we look at -- methodology paper material.")



=== CROSS-STEP FULL-PANEL TABLE ===

                    experiment                  group    mse      r2  pearson  demeaned_pearson  top1_acc  avg_rank
               brain-vol -> SC                anatomy 0.0053  0.0093   0.9155            0.1577    0.1026    0.8789
               brain-vol -> FC                anatomy 0.0135 -0.0220   0.8369            0.0467    0.0000    0.6326
  FC -> SC raw (manual sanity)             raw FC->SC 0.0056 -0.0235   0.9128            0.1269    0.1538    0.8533
  SC -> FC raw (manual sanity)             raw SC->FC 0.0140 -0.0622   0.8307            0.0848    0.0308    0.7124
  FC -> SC_residual (PLS, 3.5)       resid FC->SC PLS 0.0055 -0.0492   0.0778            0.0778    0.0718    0.8127
  SC -> FC_residual (PLS, 3.6)       resid SC->FC PLS 0.0143 -0.0705   0.0560            0.0560    0.0154    0.6487
FC -> SC_residual (PLS, 5.1 A) resid FC->SC PLS (5.1) 0.0055 -0.0494   0.0777            0.0777    0.0821    0.8125
SC -> FC_residual (PLS, 5.1 A) res